<a href="https://colab.research.google.com/github/nellyagain/30-tix-chart-data/blob/main/yahoo_weekly_movers_v4_(17).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weekly 5%+ Movers — Technical Analysis Pipeline v4.2

## How to Run

**Set `MODE` in Cell 2, then hit Runtime → Run All (Ctrl+F9). That's it.**

| Mode | When | What Runs | Time |
|------|------|-----------|------|
| `MODE = "weekly"` | Saturday/Sunday | Full rebuild + scoring + export + Drive snapshots | ~12 min |
| `MODE = "daily"` | Mon-Fri | Fast price refresh + monitor vs weekend baseline | ~4 min |

**Weekend:** Set `MODE = "weekly"` → Run All → review output → files auto-download.

**Weekday:** Set `MODE = "daily"` → Run All → read the monitor report in Cell 13.

---

**Features:** Market regime, tiered watchlist (A/B+/B/C), industry group thrust, failure risk score, fundamental flag, persistence tracking, position sizing, approaching breakout scanner, daily monitor.

**Config:** ACCOUNT_SIZE, RISK_PER_TRADE, MAX_POSITION_PCT, MODEL_VERSION — all in Cell 2.


In [1]:
# Pin versions compatible with Colab
%pip install -q "yfinance>=0.2.40" "pandas<3" "numpy<2.1" tqdm lxml html5lib


In [2]:
import time
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm
from IPython.display import display, HTML
from datetime import datetime, timedelta

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from yfinance import EquityQuery as EQ
    HAS_EQ = True
except Exception:
    HAS_EQ = False

# ===========================================================================
# RUN MODE — Set this before hitting "Run All"
# ===========================================================================
MODE = "weekly"   # "weekly" = full rebuild + export + snapshots
                  # "daily"  = fast monitor vs weekend baseline only

# ===========================================================================
# CONFIG
# ===========================================================================
MODEL_VERSION        = "v4_3"     # bump when scoring logic changes materially
RUN_DATE             = pd.Timestamp.today().normalize()

MIN_UNIVERSE_SIZE    = 500
TARGET_UNIVERSE_SIZE = 600
MIN_PRICE            = 5
MIN_DAY_VOLUME       = 200_000
MIN_WEEKLY_RETURN    = 0.05       # +5 pct over last 5 trading days
PRICE_HISTORY_PERIOD = "13mo"     # slightly over 1y to guarantee 252+ rows
MIN_HISTORY_ROWS     = 200        # minimum valid trading days
RS_LOOKBACK          = 63         # about 3 months for Mansfield RS
STAGE_SMA_PERIOD     = 150        # about 30 weeks for Weinstein stage
BASE_LOOKBACK        = 50         # days to evaluate consolidation quality
VOL_RATIO_PERIOD     = 50         # days for up/down volume ratio

# POSITION SIZING
ACCOUNT_SIZE         = 100_000    # total account value in USD
RISK_PER_TRADE       = 0.01       # risk 1% of account per trade
MAX_POSITION_PCT     = 0.20       # max 20% of account in one name

# ===========================================================================
# SECTOR ETF MAPPING
# ===========================================================================
SECTOR_ETF_MAP = {
    # GICS names (Wikipedia S&P 500 table)
    "Information Technology": "XLK",
    "Health Care": "XLV",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Communication Services": "XLC",
    "Real Estate": "XLRE",
    # Yahoo Finance variant names
    "Technology": "XLK",
    "Healthcare": "XLV",
    "Consumer Cyclical": "XLY",
    "Consumer Defensive": "XLP",
    "Financial Services": "XLF",
    "Basic Materials": "XLB",
    # Common to both
    "Energy": "XLE",
    "Financials": "XLF",
    "Industrials": "XLI",
    "Materials": "XLB",
    "Utilities": "XLU",
}
SECTOR_ETFS = sorted(set(SECTOR_ETF_MAP.values()))

# ===========================================================================
# UTILITY HELPERS
# ===========================================================================
def dedupe_keep_order(seq):
    seen = set()
    return [x for x in seq if x and x not in seen and not seen.add(x)]

def safe_float(x):
    try:
        return float(x) if x is not None else np.nan
    except Exception:
        return np.nan

def pct_rank(value, series):
    """Percentile rank of value within series (0-100)."""
    s = series.dropna()
    if len(s) == 0 or pd.isna(value):
        return np.nan
    return round(100.0 * (s < value).sum() / len(s), 1)

def top_counts(series, n=15):
    s = series.dropna().astype(str)
    if s.empty:
        return pd.DataFrame(columns=["value", "count"])
    out = s.value_counts().head(n).reset_index()
    out.columns = ["value", "count"]
    return out

def position_size(price, stop_price, account=None, risk_pct=None, max_pct=None):
    """
    Compute position size based on ATR stop and fixed-risk model.
    Returns dict with risk-based sizing, cap-aware final sizing, and actual risk %.
    """
    account = account or ACCOUNT_SIZE
    risk_pct = risk_pct or RISK_PER_TRADE
    max_pct = max_pct or MAX_POSITION_PCT

    out = {"shares": np.nan, "position_value": np.nan, "pct_of_account": np.nan,
           "actual_risk_pct": np.nan, "capped": False}

    if pd.isna(price) or pd.isna(stop_price) or price <= 0 or stop_price <= 0:
        return out

    risk_per_share = price - stop_price
    if risk_per_share <= 0:
        return out

    dollar_risk = account * risk_pct
    shares_by_risk = int(dollar_risk / risk_per_share)
    max_shares = int((account * max_pct) / price)

    if max_shares < shares_by_risk:
        shares = max_shares
        capped = True
    else:
        shares = shares_by_risk
        capped = False

    if shares <= 0:
        out.update({"shares": 0, "position_value": 0, "pct_of_account": 0, "actual_risk_pct": 0, "capped": False})
        return out

    position_value = shares * price
    actual_risk = (shares * risk_per_share) / account

    out.update({
        "shares": shares,
        "position_value": position_value,
        "pct_of_account": position_value / account,
        "actual_risk_pct": actual_risk,
        "capped": capped,
    })
    return out

# ===========================================================================
# TECHNICAL INDICATORS
# ===========================================================================
def rsi(series, period=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    avg_up = up.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_down = down.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_up / avg_down.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def atr(df, period=14):
    prev_close = df["Close"].shift(1)
    tr = pd.concat([
        (df["High"] - df["Low"]).abs(),
        (df["High"] - prev_close).abs(),
        (df["Low"]  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def atr_pct(df, period=14):
    return atr(df, period) / df["Close"]

def macd(close):
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    line = ema12 - ema26
    signal = line.ewm(span=9, adjust=False).mean()
    return line, signal, line - signal

# ===========================================================================
# RELATIVE STRENGTH
# ===========================================================================
def mansfield_rs(stock_close, bench_close, period=63):
    """
    Mansfield Relative Strength: stock/benchmark ratio
    expressed as pct change from its SMA. Positive = outperforming.
    """
    ratio = stock_close / bench_close
    ratio_sma = ratio.rolling(period).mean()
    mrs = ((ratio / ratio_sma) - 1) * 100
    return mrs

def rs_trend_direction(mrs_series):
    """
    Classify RS trend as rising, flat, or declining by comparing
    current RS to 10d and 20d ago.

    Returns: (code, label)
      +1 / 'Rising'    : RS accelerating upward
       0 / 'Flat'      : no clear direction
      -1 / 'Declining' : RS deteriorating
    """
    mrs_clean = mrs_series.dropna()
    if len(mrs_clean) < 21:
        return np.nan, ""

    now = mrs_clean.iloc[-1]
    d10 = mrs_clean.iloc[-11]
    d20 = mrs_clean.iloc[-21]

    short_delta = now - d10
    long_delta  = now - d20

    if long_delta > 0.5 and short_delta >= 0:
        return 1, "Rising"
    elif long_delta < -0.5 and short_delta <= 0:
        return -1, "Declining"
    else:
        return 0, "Flat"

def sector_adjusted_rs(stock_mrs, sector_etf_mrs):
    """
    Stock Mansfield RS minus its sector ETF Mansfield RS.
    Positive = outperforming its own sector, not just riding the wave.
    """
    if pd.isna(stock_mrs) or pd.isna(sector_etf_mrs):
        return np.nan
    return stock_mrs - sector_etf_mrs

# ===========================================================================
# STAGE CLASSIFICATION
# ===========================================================================
def classify_stage(close_price, sma_value, sma_slope):
    """
    Weinstein stage classification:
      Stage 1 (Basing):     Price below SMA, SMA flattening or turning up
      Stage 2 (Advancing):  Price above SMA, SMA rising
      Stage 3 (Topping):    Price above SMA, SMA flattening or turning down
      Stage 4 (Declining):  Price below SMA, SMA falling
    """
    if pd.isna(close_price) or pd.isna(sma_value) or pd.isna(sma_slope):
        return np.nan, ""

    above = close_price > sma_value
    flat_threshold = 0.0003

    if above and sma_slope > flat_threshold:
        return 2, "Stage 2 - Advancing"
    elif above and sma_slope <= flat_threshold and sma_slope >= -flat_threshold:
        return 3, "Stage 3 - Topping"
    elif above and sma_slope < -flat_threshold:
        return 3, "Stage 3 - Topping (SMA declining)"
    elif not above and sma_slope < -flat_threshold:
        return 4, "Stage 4 - Declining"
    elif not above and abs(sma_slope) <= flat_threshold:
        return 1, "Stage 1 - Basing"
    elif not above and sma_slope > flat_threshold:
        return 1, "Stage 1 - Basing (below rising SMA)"
    return np.nan, ""

# ===========================================================================
# VOLUME ANALYSIS
# ===========================================================================
def up_down_volume_ratio(close, volume, period=50):
    """
    Ratio of volume on up-days vs down-days.
    >1.0 = accumulation, <1.0 = distribution.
    """
    changes = close.diff()
    up_vol = volume.where(changes > 0, 0).rolling(period).sum()
    dn_vol = volume.where(changes < 0, 0).rolling(period).sum()
    return up_vol / dn_vol.replace(0, np.nan)

def breakout_day_volume(volume_series, lookback=5, avg_period=50):
    """
    Peak single-day volume in the last `lookback` days divided by
    the 50-day average volume. Captures whether the breakout day
    itself had institutional-level participation.
    Returns: ratio (e.g. 2.5 = peak day was 2.5x the 50d avg)
    """
    if len(volume_series) < avg_period:
        return np.nan
    avg_vol = volume_series.iloc[-avg_period:].mean()
    if avg_vol <= 0 or pd.isna(avg_vol):
        return np.nan
    peak_vol = volume_series.iloc[-lookback:].max()
    return peak_vol / avg_vol

# ===========================================================================
# BASE QUALITY
# ===========================================================================
def base_quality(close, lookback=50):
    """
    Measures consolidation quality:
    - depth_pct: max drawdown from peak within lookback
    - tightness: last 10d range / full lookback range (lower = tighter)
    - vol_contraction: stdev of last 10d / stdev of first 10d (VCP signature)
    """
    if len(close) < lookback:
        return np.nan, np.nan, np.nan

    recent = close.iloc[-lookback:]
    peak = recent.max()
    trough = recent.min()
    full_range = peak - trough
    depth = full_range / peak if peak > 0 else np.nan

    last10 = close.iloc[-10:]
    last10_range = last10.max() - last10.min()
    tightness = last10_range / full_range if full_range > 0 else np.nan

    early_std = close.iloc[-lookback:-lookback+10].std()
    late_std = last10.std()
    contraction = late_std / early_std if early_std > 0 else np.nan

    return depth, tightness, contraction

def weekly_closing_range(weekly_close, weekly_high, weekly_low):
    """
    Where the weekly close sits within the week's range (0.0 to 1.0).
    1.0 = closed at the high, 0.0 = closed at the low.
    Require >= 0.40 for demand confirmation (Wyckoff close-of-bar read).
    """
    rng = weekly_high - weekly_low
    if pd.isna(rng) or rng <= 0:
        return np.nan
    return (weekly_close - weekly_low) / rng

# ===========================================================================
# SCORING AND TIERING
# ===========================================================================
def breakout_quality_score(row):
    """
    Composite score (0-17) for actionability:
    +2  Stage 2
    +2  RS percentile > 85 (or +1 if > 70)
    +1  RS trend rising
    +1  20d breakout
    +1  50d breakout
    +1  Breakout day volume >= 2x (peak day in last 5d vs 50d avg)
    +1  Up/down volume ratio > 1.0
    +1  Near ATH (within 5 pct of 52w high)
    +1  Volatility contracting (VCP signature)
    +1  Weekly closing range >= 40 pct
    +1  Weekly closing range >= 70 pct (strong close bonus)
    +1  Sector-adjusted RS positive (outperforming own sector)
    +1  Industry group thrust >= 70 (broad institutional sponsorship)
    +1  Group health >= 75 (structurally strong industry)
    -1  Group health < 45 (structurally weak industry)
    """
    score = 0
    if row.get("stage_num") == 2:
        score += 2
    rs_pct = row.get("rs_percentile", 0)
    if pd.notna(rs_pct):
        if rs_pct > 85:
            score += 2
        elif rs_pct > 70:
            score += 1
    if row.get("rs_trend_code") == 1:
        score += 1
    if row.get("breakout_20d"):
        score += 1
    if row.get("breakout_50d"):
        score += 1
    bdv = row.get("breakout_day_vol", np.nan)
    if pd.notna(bdv) and bdv >= 2.0:
        score += 1
    if pd.notna(row.get("ud_volume_ratio")) and row["ud_volume_ratio"] > 1.0:
        score += 1
    if pd.notna(row.get("pct_of_52w_high")) and row["pct_of_52w_high"] >= 0.95:
        score += 1
    if pd.notna(row.get("vol_contraction")) and row["vol_contraction"] < 0.7:
        score += 1
    wcr = row.get("weekly_close_range", np.nan)
    if pd.notna(wcr):
        if wcr >= 0.70:
            score += 2
        elif wcr >= 0.40:
            score += 1
    sect_rs = row.get("sector_adj_rs", np.nan)
    if pd.notna(sect_rs) and sect_rs > 0:
        score += 1
    # Group sponsorship: industry thrust confirms institutional breadth
    gt = row.get("group_thrust", np.nan)
    if pd.notna(gt) and gt >= 70:
        score += 1
    gh = row.get("group_health_score", np.nan)
    if pd.notna(gh):
        if gh >= 75:
            score += 1
        elif gh < 45:
            score -= 1
    return score

def assign_tier(row):
    """
    A   : Stage 2 + BQ >= 7 + RS > 70 + near ATH + WCR >= 40% + RS not declining
    B+  : Stage 3 Watch: Stage 3 + near ATH (>= 90%) + RS > 80 + WCR >= 40%
          (SMA flattening but price still strong - watch for Stage 2 re-entry)
    B   : Stage 1 or 2 + BQ >= 4
    C   : everything else
    """
    stage = row.get("stage_num", np.nan)
    bq = row.get("breakout_quality", 0)
    rs = row.get("rs_percentile", 0)
    near_ath = pd.notna(row.get("pct_of_52w_high")) and row["pct_of_52w_high"] >= 0.95
    wcr_ok = pd.notna(row.get("weekly_close_range")) and row["weekly_close_range"] >= 0.40
    rs_code = row.get("rs_trend_code", np.nan)
    rs_not_declining = pd.isna(rs_code) or rs_code >= 0

    if stage == 2 and bq >= 7 and pd.notna(rs) and rs > 70 and near_ath and wcr_ok and rs_not_declining:
        return "A"

    pct_high = row.get("pct_of_52w_high", 0) or 0
    if stage == 3 and pd.notna(rs) and rs > 80 and pct_high >= 0.90 and wcr_ok:
        return "B+"

    if stage in [1, 2] and bq >= 4:
        return "B"

    return "C"

# ===========================================================================
# PRIORITY SCORING
# ===========================================================================
PRIORITY_TOP_N = 8   # how many names in the final priority list

def priority_score(row, regime_verdict, concentration_penalty=0):
    """
    Single priority score combining all signals for final ranking.
    Higher = higher priority. Concentration penalty is applied externally
    by the greedy selection algorithm.
    """
    score = 0.0

    # Tier points: A > New Setup > B+ > B
    status = row.get("_status", "")
    tier = row.get("tier", "C")
    if tier == "A":
        score += 12
    elif status == "NEW_SETUP":
        score += 10
    elif tier == "B+":
        score += 8
    elif tier == "B":
        score += 5

    # Breakout quality (0-15, weight 1.5)
    bq = row.get("breakout_quality", 0)
    if pd.notna(bq):
        score += 1.5 * bq

    # RS percentile (0-100, weight 0.08)
    rs = row.get("rs_percentile", 0)
    if pd.notna(rs):
        score += 0.08 * rs

    # Sector-adjusted RS bonus
    sa_rs = row.get("sector_adj_rs", 0)
    if pd.notna(sa_rs) and sa_rs > 5:
        score += min(3.0, sa_rs * 0.1)

    # Group thrust (0-100, weight 0.05)
    gt = row.get("group_thrust", 0)
    if pd.notna(gt):
        score += 0.05 * gt

    # Failure risk penalty (0-10, weight -0.75)
    fr = row.get("failure_risk", 0)
    if pd.notna(fr):
        score -= 0.75 * fr

    # Weekly close range (0-1, weight 2.0)
    wcr = row.get("weekly_close_range", 0)
    if pd.notna(wcr):
        score += 2.0 * wcr

    # U/D volume ratio (capped at 2.5, weight 1.0)
    ud = row.get("ud_volume_ratio", 0)
    if pd.notna(ud):
        score += 1.0 * min(ud, 2.5)

    # Fundamental flag
    ff = row.get("fund_flag", "")
    if ff == "FUND_OK":
        score += 2.0
    elif ff == "FUND_MIXED":
        score += 0.5
    elif ff == "FUND_WEAK":
        score -= 1.0

    # Fresh/new bonus
    pl = row.get("persistence_label", "")
    if isinstance(pl, str):
        if "Persistent" in pl:
            score += 1.5
        elif pl == "Fresh" or pl == "New Setup":
            score += 0.5
        elif pl == "Fading":
            score -= 2.0

    # Extension penalty: already moved 15%+ this week
    ret = row.get("weekly_return", 0)
    if pd.notna(ret) and ret > 0.15:
        score -= 2.0 * (ret - 0.15) / 0.10  # -2 per 10% above 15%

    # Group health boost
    gh = row.get("group_health_score", 0)
    if pd.notna(gh):
        score += 0.08 * gh

    # Concentration penalty (applied by greedy selector, not here)
    score -= concentration_penalty

    # Regime adjustment: in MIXED/RISK-OFF, penalize lower-quality names more
    if regime_verdict in ["MIXED / TRANSITIONAL", "RISK-OFF"]:
        if bq and pd.notna(bq) and bq < 8:
            score -= 1.5
        if tier in ["B", "B+"]:
            score -= 1.0

    return round(score, 2)

# ===========================================================================
# FAILURE RISK + FUNDAMENTAL FLAGS
# ===========================================================================
def failure_risk_score(row):
    """
    How likely is this position to fail? (0-10, higher = more danger)
    +2  Stage 3 or 4
    +1  RSI > 80 (overbought)
    +1  Extended: near ATH + RSI > 75
    +1  U/D volume < 0.9 (distribution)
    +1  Weekly close in bottom 30% of range
    +1  RS trend declining
    +1  Sector-adjusted RS negative
    +1  Base depth > 40% (deep correction)
    +1  Breakout day volume < 1.2x (no conviction)
    """
    score = 0
    stage = row.get("stage_num", np.nan)
    if stage in [3, 4]:
        score += 2

    rsi_val = row.get("rsi14", 0)
    if pd.notna(rsi_val) and rsi_val > 80:
        score += 1

    pct_high = row.get("pct_of_52w_high", 0)
    if pd.notna(pct_high) and pct_high >= 0.98 and pd.notna(rsi_val) and rsi_val > 75:
        score += 1

    ud = row.get("ud_volume_ratio", np.nan)
    if pd.notna(ud) and ud < 0.9:
        score += 1

    wcr = row.get("weekly_close_range", np.nan)
    if pd.notna(wcr) and wcr < 0.30:
        score += 1

    if row.get("rs_trend_code") == -1:
        score += 1

    sa_rs = row.get("sector_adj_rs", np.nan)
    if pd.notna(sa_rs) and sa_rs < 0:
        score += 1

    depth = row.get("base_depth", np.nan)
    if pd.notna(depth) and depth > 0.40:
        score += 1

    bdv = row.get("breakout_day_vol", np.nan)
    if pd.notna(bdv) and bdv < 1.2:
        score += 1

    return min(10, score)

def failure_risk_label(score):
    if score <= 2:
        return "Low Risk"
    elif score <= 4:
        return "Moderate"
    elif score <= 6:
        return "Elevated"
    else:
        return "High Risk"

# ===========================================================================
# yf.download() COLUMN NORMALIZER
# ===========================================================================
def normalize_download(px, tickers_list):
    """Handle both (Price, Ticker) and (Ticker, Price) MultiIndex layouts."""
    if not isinstance(px.columns, pd.MultiIndex):
        result = {}
        for field in ["Close", "High", "Low", "Volume"]:
            if field in px.columns:
                t = tickers_list[0] if len(tickers_list) == 1 else "UNKNOWN"
                result[field] = px[[field]].rename(columns={field: t})
        return result

    level0_vals = px.columns.get_level_values(0).unique().tolist()
    price_fields = {"Close", "High", "Low", "Open", "Volume", "Adj Close"}

    if set(level0_vals) & price_fields:
        print("  Column layout: (Price, Ticker)")
        return {f: px[f].copy() for f in ["Close", "High", "Low", "Volume"] if f in level0_vals}
    else:
        print("  Column layout: (Ticker, Price)")
        result = {}
        available = [t for t in level0_vals if t in set(tickers_list)]
        for field in ["Close", "High", "Low", "Volume"]:
            cols = {}
            for t in available:
                try:
                    cols[t] = px[(t, field)]
                except KeyError:
                    pass
            if cols:
                result[field] = pd.DataFrame(cols, index=px.index)
        return result

print("Helpers loaded")


Helpers loaded


In [3]:
# ===========================================================================
# PERSISTENCE LAYER: GOOGLE DRIVE STORAGE
# ===========================================================================
from google.colab import drive
import os

drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/leadership_monitor"
SNAP_DIR = f"{BASE}/snapshots"
GROUP_DIR = f"{BASE}/group_snapshots"

for d in [BASE, SNAP_DIR, GROUP_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Persistence storage ready: {BASE}")
print(f"  Ticker snapshots: {SNAP_DIR}")
print(f"  Group snapshots: {GROUP_DIR}")

# Count existing snapshots
existing_snaps = [f for f in os.listdir(SNAP_DIR) if f.endswith('.csv')] if os.path.exists(SNAP_DIR) else []
existing_groups = [f for f in os.listdir(GROUP_DIR) if f.endswith('.csv')] if os.path.exists(GROUP_DIR) else []
print(f"  Existing ticker snapshots: {len(existing_snaps)}")
print(f"  Existing group snapshots: {len(existing_groups)}")


Mounted at /content/drive
Persistence storage ready: /content/drive/MyDrive/leadership_monitor
  Ticker snapshots: /content/drive/MyDrive/leadership_monitor/snapshots
  Group snapshots: /content/drive/MyDrive/leadership_monitor/group_snapshots
  Existing ticker snapshots: 1
  Existing group snapshots: 1


In [4]:
# ===========================================================================
# BUILD UNIVERSE
# ===========================================================================
def get_universe(target_size=600, min_size=500):
    symbols = []

    if HAS_EQ:
        try:
            print("Building universe from Yahoo screener...")
            q = EQ("and", [
                EQ("eq", ["region", "us"]),
                EQ("is-in", ["exchange", "NMS", "NYQ", "ASE"]),
                EQ("gte", ["intradayprice", MIN_PRICE]),
                EQ("gte", ["dayvolume", MIN_DAY_VOLUME]),
            ])
            for offset in [0, 250, 500, 750]:
                resp = yf.screen(q, offset=offset, size=250,
                                 sortField="intradaymarketcap", sortAsc=False)
                page = [x.get("symbol") for x in resp.get("quotes", [])
                        if isinstance(x, dict) and x.get("symbol")]
                if not page:
                    break
                page = [s for s in page if isinstance(s, str)
                        and "^" not in s and "/" not in s and "=" not in s]
                symbols.extend(page)
                time.sleep(0.5)
            symbols = dedupe_keep_order(symbols)
            if len(symbols) >= min_size:
                print(f"  Yahoo screener: {len(symbols)} names")
                return symbols[:target_size]
            print(f"  Yahoo screener returned only {len(symbols)}, falling back...")
        except Exception as e:
            print(f"  Yahoo screener failed: {e}")

    print("Building fallback universe from S&P 500 + Nasdaq-100...")
    sp500 = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
    sp500_sym = sp500["Symbol"].astype(str).str.strip().str.replace(".", "-", regex=False).tolist()
    try:
        ndx_tables = pd.read_html("https://en.wikipedia.org/wiki/Nasdaq-100")
        ndx_sym = []
        for tbl in ndx_tables:
            cols_lower = [str(c).strip().lower() for c in tbl.columns]
            for orig, low in zip(tbl.columns, cols_lower):
                if low in ("ticker", "symbol"):
                    ndx_sym = tbl[orig].astype(str).str.strip().str.replace(".", "-", regex=False).tolist()
                    break
            if ndx_sym:
                break
    except Exception:
        ndx_sym = []
    symbols = dedupe_keep_order(sp500_sym + ndx_sym)
    print(f"  Fallback universe: {len(symbols)} names")
    return symbols[:max(target_size, min_size)]

universe = get_universe(TARGET_UNIVERSE_SIZE, MIN_UNIVERSE_SIZE)
print(f"Universe: {len(universe)} tickers")

# ===========================================================================
# BUILD SECTOR MAP FROM S&P 500 TABLE
# ===========================================================================
print("\nBuilding sector + industry maps from S&P 500 constituents...")
sector_map = {}
industry_map = {}
try:
    sp500_tbl = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]
    # Robust column detection
    sector_col, industry_col = None, None
    for col in sp500_tbl.columns:
        cl = str(col).lower().strip()
        if "sector" in cl and sector_col is None:
            sector_col = col
        if "sub" in cl and "industr" in cl:
            industry_col = col
    if industry_col is None:
        for col in sp500_tbl.columns:
            if "industr" in str(col).lower() and col != sector_col:
                industry_col = col
                break
    print(f"  Columns detected — sector: '{sector_col}', industry: '{industry_col}'")
    print(f"  All columns: {sp500_tbl.columns.tolist()}")

    for _, row in sp500_tbl.iterrows():
        sym = str(row.get("Symbol", "")).strip().replace(".", "-")
        sect = str(row.get(sector_col, "")).strip() if sector_col else ""
        ind = str(row.get(industry_col, "")).strip() if industry_col else ""
        if sym and sect and sect != "nan" and sect != "":
            sector_map[sym] = sect
        if sym and ind and ind != "nan" and ind != "":
            industry_map[sym] = ind
    print(f"  Sector map: {len(sector_map)} names")
    print(f"  Industry map: {len(industry_map)} names")
    if len(industry_map) < 100:
        print(f"  WARNING: Industry map smaller than expected — check column detection")
except Exception as e:
    print(f"  Could not build maps: {e}")

# For non-S&P names, do a lightweight yfinance sector lookup (winners only, later)
# This avoids 600 API calls for the full universe

# ===========================================================================
# DOWNLOAD PRICE DATA (universe + SPY + sector ETFs)
# ===========================================================================
all_tickers = dedupe_keep_order(universe + ["SPY"] + SECTOR_ETFS)

print(f"\nDownloading {PRICE_HISTORY_PERIOD} of daily data for {len(all_tickers)} tickers...")
print(f"  (includes SPY + {len(SECTOR_ETFS)} sector ETFs: {', '.join(SECTOR_ETFS)})")
px = yf.download(
    tickers=all_tickers,
    period=PRICE_HISTORY_PERIOD,
    interval="1d",
    auto_adjust=True,
    progress=False,
    threads=True,
    timeout=30
)

fields = normalize_download(px, all_tickers)
for f in ["Close", "High", "Low", "Volume"]:
    if f not in fields:
        raise RuntimeError(f"Missing field: {f}")

close  = fields["Close"].copy()
high   = fields["High"].copy()
low    = fields["Low"].copy()
volume = fields["Volume"].copy()

# Validate benchmark data
benchmarks = ["SPY"] + SECTOR_ETFS
for bm in benchmarks:
    if bm not in close.columns or close[bm].dropna().shape[0] < MIN_HISTORY_ROWS:
        print(f"  Warning: {bm} data missing or insufficient")

spy_close = close["SPY"].dropna()

# Extract sector ETF close prices for RS computation
sector_etf_close = {}
for etf in SECTOR_ETFS:
    if etf in close.columns and close[etf].dropna().shape[0] >= MIN_HISTORY_ROWS:
        sector_etf_close[etf] = close[etf].dropna()
print(f"  Sector ETFs with valid data: {len(sector_etf_close)}/{len(SECTOR_ETFS)}")

# Validate universe tickers
valid = [t for t in close.columns
         if t not in (["SPY"] + SECTOR_ETFS)
         and close[t].dropna().shape[0] >= MIN_HISTORY_ROWS
         and volume[t].dropna().shape[0] >= MIN_HISTORY_ROWS]

keep_cols = valid + ["SPY"] + [e for e in SECTOR_ETFS if e in close.columns]
close  = close[[c for c in keep_cols if c in close.columns]]
high   = high[[c for c in valid if c in high.columns]]
low    = low[[c for c in valid if c in low.columns]]
volume = volume[[c for c in valid if c in volume.columns]]

print(f"\nValid tickers with {MIN_HISTORY_ROWS}+ days: {len(valid)}")
print(f"Date range: {close.index[0].strftime('%Y-%m-%d')} to {close.index[-1].strftime('%Y-%m-%d')}")
print(f"Trading days: {len(close)}")


Building universe from Yahoo screener...
  Yahoo screener: 1000 names
Universe: 600 tickers

Building sector + industry maps from S&P 500 constituents...
  Could not build maps: HTTP Error 403: Forbidden

  (includes SPY + 11 sector ETFs: XLB, XLC, XLE, XLF, XLI, XLK, XLP, XLRE, XLU, XLV, XLY)
  Column layout: (Price, Ticker)
  Sector ETFs with valid data: 11/11

Valid tickers with 200+ days: 596
Date range: 2025-02-07 to 2026-03-06
Trading days: 270


In [5]:
# ===========================================================================
# MARKET REGIME DASHBOARD
# ===========================================================================
print("Computing market regime...\n")

regime = {}

# --- SPY trend context ---
spy = close["SPY"]
spy_sma50  = spy.rolling(50).mean().iloc[-1]
spy_sma200 = spy.rolling(200).mean().iloc[-1]
spy_last   = spy.iloc[-1]
spy_rsi14  = rsi(spy, 14).iloc[-1]
spy_ret_5d  = (spy.iloc[-1] / spy.iloc[-6] - 1) * 100 if len(spy) >= 6 else np.nan
spy_ret_20d = (spy.iloc[-1] / spy.iloc[-21] - 1) * 100 if len(spy) >= 21 else np.nan

regime["SPY_close"]      = round(spy_last, 2)
regime["SPY_vs_50sma"]   = "ABOVE" if spy_last > spy_sma50 else "BELOW"
regime["SPY_vs_200sma"]  = "ABOVE" if spy_last > spy_sma200 else "BELOW"
regime["SPY_50_vs_200"]  = "BULLISH (golden cross)" if spy_sma50 > spy_sma200 else "BEARISH (death cross)"
regime["SPY_RSI14"]      = round(spy_rsi14, 1)
regime["SPY_5d_return"]  = f"{spy_ret_5d:+.2f}%"
regime["SPY_20d_return"] = f"{spy_ret_20d:+.2f}%"

# --- Breadth measures across universe ---
last_close = close[valid].iloc[-1]
sma50_all  = close[valid].rolling(50).mean().iloc[-1]
sma200_all = close[valid].rolling(200).mean().iloc[-1]
high_52w   = close[valid].rolling(min(252, len(close))).max().iloc[-1]
low_52w    = close[valid].rolling(min(252, len(close))).min().iloc[-1]

pct_above_50  = round(100 * (last_close > sma50_all).mean(), 1)
pct_above_200 = round(100 * (last_close > sma200_all).mean(), 1)
pct_near_high = round(100 * ((last_close / high_52w) > 0.90).mean(), 1)
pct_in_bear   = round(100 * ((last_close / high_52w) < 0.60).mean(), 1)

regime["pct_above_50sma"]  = f"{pct_above_50}%"
regime["pct_above_200sma"] = f"{pct_above_200}%"
regime["pct_within_10pct_of_52w_high"] = f"{pct_near_high}%"
regime["pct_more_than_40pct_off_high"] = f"{pct_in_bear}%"

# --- Advance / Decline (5-day) ---
if len(close) >= 6:
    ret_5d_all = close[valid].iloc[-1] / close[valid].iloc[-6] - 1
    advancers = int((ret_5d_all > 0).sum())
    decliners = int((ret_5d_all < 0).sum())
    regime["5d_advancers"] = advancers
    regime["5d_decliners"] = decliners
    regime["5d_AD_ratio"]  = round(advancers / max(decliners, 1), 2)

# --- Net new highs ---
at_high = int((last_close / high_52w >= 0.98).sum())
at_low  = int((last_close / low_52w <= 1.02).sum())
regime["stocks_at_52w_high"] = at_high
regime["stocks_at_52w_low"]  = at_low
regime["net_new_highs"]      = at_high - at_low

# --- Universe median RSI ---
rsi_all = pd.Series({t: rsi(close[t].dropna(), 14).iloc[-1] for t in valid})
regime["universe_median_RSI"] = round(rsi_all.median(), 1)

# --- Regime verdict ---
if pct_above_200 > 60 and spy_last > spy_sma200 and pct_above_50 > 50:
    verdict = "RISK-ON"
elif pct_above_200 < 40 and spy_last < spy_sma200:
    verdict = "RISK-OFF"
else:
    verdict = "MIXED / TRANSITIONAL"
regime["REGIME_VERDICT"] = verdict

# --- Display ---
print("=" * 64)
print("  MARKET REGIME DASHBOARD")
print("=" * 64)
for k, v in regime.items():
    print(f"  {k:>42s} : {v}")
print("=" * 64)

# Store for export
regime_df = pd.DataFrame([regime])


Computing market regime...

  MARKET REGIME DASHBOARD
                                   SPY_close : 672.38
                                SPY_vs_50sma : BELOW
                               SPY_vs_200sma : ABOVE
                               SPY_50_vs_200 : BULLISH (golden cross)
                                   SPY_RSI14 : 38.5
                               SPY_5d_return : -1.98%
                              SPY_20d_return : -0.77%
                             pct_above_50sma : 46.1%
                            pct_above_200sma : 63.6%
                pct_within_10pct_of_52w_high : 38.9%
                pct_more_than_40pct_off_high : 5.0%
                                5d_advancers : 157
                                5d_decliners : 438
                                 5d_AD_ratio : 0.36
                          stocks_at_52w_high : 64
                           stocks_at_52w_low : 9
                               net_new_highs : 55
                         universe_median_R

In [6]:
# ===========================================================================
# FIND WEEKLY WINNERS
# ===========================================================================
print(f"Scanning for {MIN_WEEKLY_RETURN:.0%}+ weekly movers...\n")

if len(close) < 6:
    raise RuntimeError("Not enough rows to compute weekly returns.")

weekly_return = close[valid].iloc[-1] / close[valid].iloc[-6] - 1
avg_dollar_vol = (close[valid] * volume[valid]).rolling(20).mean().iloc[-1]

winners = pd.DataFrame({
    "ticker": weekly_return.index,
    "weekly_return": weekly_return.values,
    "avg_dollar_vol_20d": avg_dollar_vol.reindex(weekly_return.index).values
})
winners = winners[
    (winners["weekly_return"] >= MIN_WEEKLY_RETURN) &
    (winners["avg_dollar_vol_20d"] > 0)
].sort_values("weekly_return", ascending=False).reset_index(drop=True)

winner_tickers = winners["ticker"].tolist()

print(f"Winners: {len(winner_tickers)} stocks up >= {MIN_WEEKLY_RETURN:.0%} this week")
print(f"(out of {len(valid)} universe names)\n")

if len(winner_tickers) == 0:
    print("No winners found this week. Try lowering MIN_WEEKLY_RETURN in config.")
else:
    display(winners.style.format({
        "weekly_return": "{:.1%}",
        "avg_dollar_vol_20d": "${:,.0f}"
    }).set_caption(f"Weekly {MIN_WEEKLY_RETURN:.0%}+ Movers"))


Scanning for 5%+ weekly movers...

Winners: 57 stocks up >= 5% this week
(out of 596 universe names)



,ticker,weekly_return,avg_dollar_vol_20d
0,VG,28.8%,"$183,158,900"
1,IOT,22.4%,"$283,403,406"
2,ESLT,21.7%,"$154,285,144"
3,LYB,18.1%,"$433,566,029"
4,INTU,17.6%,"$2,226,753,545"
5,EXPE,16.0%,"$900,253,401"
6,TRI,15.6%,"$376,265,515"
7,APP,15.5%,"$3,133,246,642"
8,CRWD,15.3%,"$2,098,110,714"
9,NOW,15.1%,"$2,347,037,615"


In [7]:
# ===========================================================================
# SECTOR LOOKUP FOR NON-S&P NAMES (winners only, to limit API calls)
# ===========================================================================
# Look up sectors for winner tickers not in S&P 500 map
print("Looking up sectors for winner tickers not in S&P 500 map...")
missing_winner_data = [t for t in winner_tickers if t not in sector_map or t not in industry_map]
if missing_winner_data:
    for t in tqdm(missing_winner_data, desc="Winner sector/industry lookup"):
        try:
            tk = yf.Ticker(t)
            info = tk.get_info() if hasattr(tk, 'get_info') else (tk.info or {})
            sect = info.get("sector", "")
            ind = info.get("industry", "")
            if sect and t not in sector_map:
                sector_map[t] = sect
            if ind and t not in industry_map:
                industry_map[t] = ind
            time.sleep(0.15)
        except Exception:
            pass
    print(f"  Sector map: {len(sector_map)}, Industry map: {len(industry_map)}")

# ===========================================================================
# LIGHTWEIGHT EPS/REVENUE DIRECTION FLAGS FOR WINNERS
# ===========================================================================
print("\nFetching EPS/revenue direction for winners...")
fund_flags = {}
for t in tqdm(winner_tickers, desc="Fundamentals"):
    try:
        tk = yf.Ticker(t)
        info = {}
        try:
            info = tk.get_info() or {}
        except Exception:
            try:
                info = tk.info or {}
            except Exception:
                pass
        rev_g = info.get("revenueGrowth")
        eps_g = info.get("earningsGrowth")
        rev_ok = pd.notna(rev_g) and rev_g > 0
        rev_bad = pd.notna(rev_g) and rev_g <= 0
        eps_ok = pd.notna(eps_g) and eps_g > 0
        eps_bad = pd.notna(eps_g) and eps_g <= 0
        if rev_ok and eps_ok:
            flag = "FUND_OK"
        elif rev_bad and eps_bad:
            flag = "FUND_WEAK"
        elif (rev_ok or eps_ok) and not (rev_bad and eps_bad):
            flag = "FUND_MIXED"
        else:
            flag = "FUND_UNKNOWN"
        fund_flags[t] = {"fund_flag": flag,
                         "rev_growth": round(rev_g, 3) if pd.notna(rev_g) else np.nan,
                         "eps_growth": round(eps_g, 3) if pd.notna(eps_g) else np.nan}
        time.sleep(0.15)
    except Exception:
        fund_flags[t] = {"fund_flag": "FUND_UNKNOWN", "rev_growth": np.nan, "eps_growth": np.nan}
print(f"  Fund flags: {len(fund_flags)} tickers")

# ===========================================================================
# PRE-COMPUTE SECTOR ETF RS VALUES
# ===========================================================================
print("\nComputing sector ETF Mansfield RS vs SPY...")
sector_etf_rs_now = {}
for etf, etf_close in sector_etf_close.items():
    spy_aligned = spy_close.reindex(etf_close.index)
    mrs = mansfield_rs(etf_close, spy_aligned, RS_LOOKBACK)
    if len(mrs.dropna()) > 0:
        sector_etf_rs_now[etf] = mrs.iloc[-1]
        print(f"  {etf}: RS = {mrs.iloc[-1]:+.1f}")

# ===========================================================================
# FULL TECHNICAL PROFILE - ALL UNIVERSE NAMES
# ===========================================================================
print("\nComputing technical profiles for full universe...\n")

all_rs_values = {}
feature_rows = []

for t in tqdm(valid, desc="Profiling"):
    try:
        df = pd.DataFrame({
            "Close":  close[t],
            "High":   high[t] if t in high.columns else np.nan,
            "Low":    low[t] if t in low.columns else np.nan,
            "Volume": volume[t] if t in volume.columns else np.nan
        }).dropna()

        if len(df) < MIN_HISTORY_ROWS:
            continue

        c = df["Close"]
        v = df["Volume"]
        last = df.iloc[-1]

        # -- Moving Averages --
        sma20  = c.rolling(20).mean()
        sma50  = c.rolling(50).mean()
        sma150 = c.rolling(STAGE_SMA_PERIOD).mean()
        sma200 = c.rolling(200).mean()

        # -- RSI --
        rsi14 = rsi(c, 14)

        # -- ATR --
        atr14 = atr(df, 14)
        atr14_pct_series = atr14 / c

        # -- MACD --
        macd_line, macd_sig, macd_hist_series = macd(c)

        # -- Mansfield Relative Strength --
        spy_aligned = spy_close.reindex(c.index)
        mrs = mansfield_rs(c, spy_aligned, RS_LOOKBACK)
        mrs_now = mrs.iloc[-1] if len(mrs) > 0 and pd.notna(mrs.iloc[-1]) else np.nan
        all_rs_values[t] = mrs_now

        # -- RS Trend Direction --
        rs_code, rs_label = rs_trend_direction(mrs)

        # -- Sector-Adjusted RS --
        t_sector = sector_map.get(t, "")
        t_sector_etf = SECTOR_ETF_MAP.get(t_sector, "")
        t_sector_etf_rs = sector_etf_rs_now.get(t_sector_etf, np.nan)
        sect_adj = sector_adjusted_rs(mrs_now, t_sector_etf_rs)

        # -- Weinstein Stage --
        sma150_now = sma150.iloc[-1] if pd.notna(sma150.iloc[-1]) else np.nan
        if len(sma150.dropna()) >= 20 and pd.notna(sma150_now):
            sma150_20_ago = sma150.dropna().iloc[-20]
            sma150_slope = (sma150_now - sma150_20_ago) / (sma150_20_ago * 20) if sma150_20_ago != 0 else 0
        else:
            sma150_slope = np.nan

        stage_num, stage_label = classify_stage(last["Close"], sma150_now, sma150_slope)

        # -- Volume Analysis --
        vol20 = v.rolling(20).mean()
        vol20_now = vol20.iloc[-1]
        rel_vol = safe_float(last["Volume"] / vol20_now) if pd.notna(vol20_now) and vol20_now != 0 else np.nan
        ud_ratio = up_down_volume_ratio(c, v, VOL_RATIO_PERIOD)
        ud_now = safe_float(ud_ratio.iloc[-1]) if len(ud_ratio) > 0 else np.nan

        # -- Breakout Day Volume --
        bdv = breakout_day_volume(v, lookback=5, avg_period=50)

        # -- Base Quality --
        depth, tightness, contraction = base_quality(c, BASE_LOOKBACK)

        # -- Weekly Closing Range --
        wcr = weekly_closing_range(
            c.iloc[-1],
            df["High"].iloc[-5:].max() if len(df) >= 5 else np.nan,
            df["Low"].iloc[-5:].min() if len(df) >= 5 else np.nan
        )

        # -- Price Levels --
        n_rows = len(c)
        high20_prev = c.rolling(20).max().shift(1).iloc[-1]
        high50_prev = c.rolling(50).max().shift(1).iloc[-1]

        # Distance to pivot: how far below the 20d high (the breakout trigger level)
        # Negative = below pivot, 0 = at pivot, positive = above pivot
        pct_to_pivot = (last["Close"] / high20_prev - 1) if pd.notna(high20_prev) and high20_prev > 0 else np.nan
        high252     = c.rolling(min(252, n_rows)).max().iloc[-1]
        low252      = c.rolling(min(252, n_rows)).min().iloc[-1]

        above20  = bool(last["Close"] > sma20.iloc[-1])  if pd.notna(sma20.iloc[-1])  else np.nan
        above50  = bool(last["Close"] > sma50.iloc[-1])  if pd.notna(sma50.iloc[-1])  else np.nan
        above200 = bool(last["Close"] > sma200.iloc[-1]) if pd.notna(sma200.iloc[-1]) else np.nan
        bo20     = bool(last["Close"] >= high20_prev) if pd.notna(high20_prev) else np.nan
        bo50     = bool(last["Close"] >= high50_prev) if pd.notna(high50_prev) else np.nan
        macd_bull = bool(macd_line.iloc[-1] > macd_sig.iloc[-1]) if pd.notna(macd_sig.iloc[-1]) else np.nan

        # -- ATR-based stop levels --
        atr_now = atr14.iloc[-1]
        stop_1atr = last["Close"] - atr_now if pd.notna(atr_now) else np.nan
        stop_2atr = last["Close"] - 2 * atr_now if pd.notna(atr_now) else np.nan

        # -- Returns --
        ret_5d  = safe_float(c.pct_change(5).iloc[-1])  if n_rows > 5  else np.nan
        ret_20d = safe_float(c.pct_change(20).iloc[-1]) if n_rows > 20 else np.nan

        feature_rows.append({
            "ticker": t,
            "close": safe_float(last["Close"]),
            "sector": t_sector if t_sector else np.nan,
            "industry": industry_map.get(t, np.nan),
            "ret_5d": ret_5d,
            "ret_20d": ret_20d,
            # RS
            "mansfield_rs": safe_float(mrs_now),
            "rs_trend_code": rs_code,
            "rs_trend": rs_label,
            "sector_adj_rs": safe_float(sect_adj),
            # Stage
            "stage_num": stage_num,
            "stage_label": stage_label,
            "sma150_slope": safe_float(sma150_slope),
            # Momentum
            "rsi14": safe_float(rsi14.iloc[-1]),
            "macd_bull": macd_bull,
            # Volume
            "rel_volume_20": rel_vol,
            "ud_volume_ratio": safe_float(ud_now),
            "breakout_day_vol": safe_float(bdv),
            # Volatility
            "atr14_pct": safe_float(atr14_pct_series.iloc[-1]),
            # Price structure
            "pct_of_52w_high": safe_float(last["Close"] / high252) if pd.notna(high252) and high252 != 0 else np.nan,
            "weekly_close_range": safe_float(wcr),
            "near_ath": bool(pd.notna(high252) and high252 > 0 and last["Close"] / high252 >= 0.95),
            "above_sma20":  above20,
            "above_sma50":  above50,
            "above_sma200": above200,
            "pct_to_pivot_20d": safe_float(pct_to_pivot),
            "breakout_20d": bo20,
            "breakout_50d": bo50,
            # Base quality
            "base_depth":      safe_float(depth),
            "base_tightness":  safe_float(tightness),
            "vol_contraction": safe_float(contraction),
            # Levels
            "stop_1atr": safe_float(stop_1atr),
            "stop_2atr": safe_float(stop_2atr),
            "high_52w":  safe_float(high252),
            "low_52w":   safe_float(low252),
        })
    except Exception:
        pass

features_df = pd.DataFrame(feature_rows)

# -- RS Percentile Rank across universe --
rs_series = pd.Series(all_rs_values)
features_df["rs_percentile"] = features_df["ticker"].apply(
    lambda t: pct_rank(all_rs_values.get(t, np.nan), rs_series)
)

# -- Failure Risk Score --
features_df["failure_risk"] = features_df.apply(failure_risk_score, axis=1)
features_df["failure_risk_label"] = features_df["failure_risk"].apply(failure_risk_label)

# -- Breakout Quality Score + Tier --
features_df["breakout_quality"] = features_df.apply(breakout_quality_score, axis=1)
features_df["tier"] = features_df.apply(assign_tier, axis=1)

# -- Fund Flags (winner tickers only) --
features_df["fund_flag"] = features_df["ticker"].map(
    lambda t: fund_flags.get(t, {}).get("fund_flag", np.nan))
features_df["rev_growth"] = features_df["ticker"].map(
    lambda t: fund_flags.get(t, {}).get("rev_growth", np.nan))
features_df["eps_growth"] = features_df["ticker"].map(
    lambda t: fund_flags.get(t, {}).get("eps_growth", np.nan))

# -- Extended sector lookup for top RS names (setup candidates) --
# Now that all_rs_values is populated, find top RS names missing sector data
print("\nLooking up sectors for top RS universe names (setup candidates)...")
top_rs_missing = sorted(
    [t for t in valid if t not in sector_map or t not in industry_map],
    key=lambda t: all_rs_values.get(t, -999),
    reverse=True
)[:60]

if top_rs_missing:
    print(f"  Looking up {len(top_rs_missing)} tickers...")
    for t in tqdm(top_rs_missing, desc="Setup sector/industry lookup"):
        try:
            tk = yf.Ticker(t)
            info = tk.get_info() if hasattr(tk, 'get_info') else (tk.info or {})
            sect = info.get("sector", "")
            ind = info.get("industry", "")
            if sect and t not in sector_map:
                sector_map[t] = sect
            if ind and t not in industry_map:
                industry_map[t] = ind
            time.sleep(0.15)
        except Exception:
            pass
    print(f"  Sector map: {len(sector_map)}, Industry map: {len(industry_map)}")

# -- Backfill sector data for any tickers that got sector lookups after profiling --
for idx, row in features_df.iterrows():
    t = row["ticker"]
    if pd.isna(row.get("sector")) and t in sector_map:
        features_df.at[idx, "sector"] = sector_map[t]
    if pd.isna(row.get("industry")) and t in industry_map:
        features_df.at[idx, "industry"] = industry_map[t]
        # Recompute sector-adjusted RS
        t_sector_etf = SECTOR_ETF_MAP.get(sector_map[t], "")
        t_etf_rs = sector_etf_rs_now.get(t_sector_etf, np.nan)
        features_df.at[idx, "sector_adj_rs"] = sector_adjusted_rs(
            all_rs_values.get(t, np.nan), t_etf_rs)

# -- Winner subset --
winner_features = features_df[features_df["ticker"].isin(winner_tickers)].copy()

print(f"\nProfiled {len(features_df)} tickers")
print(f"Winner profiles: {len(winner_features)}")
print(f"Tickers with sector data: {features_df['sector'].notna().sum()}")


Looking up sectors for winner tickers not in S&P 500 map...


Winner sector/industry lookup:   0%|          | 0/57 [00:00<?, ?it/s]

  Sector map: 57, Industry map: 57

Fetching EPS/revenue direction for winners...


Fundamentals:   0%|          | 0/57 [00:00<?, ?it/s]

  Fund flags: 57 tickers

Computing sector ETF Mansfield RS vs SPY...
  XLB: RS = +4.1
  XLC: RS = +2.7
  XLE: RS = +16.8
  XLF: RS = -3.7
  XLI: RS = +5.1
  XLK: RS = -2.2
  XLP: RS = +5.9
  XLRE: RS = +5.4
  XLU: RS = +8.7
  XLV: RS = +0.1
  XLY: RS = -2.4

Computing technical profiles for full universe...



Profiling:   0%|          | 0/596 [00:00<?, ?it/s]


Looking up sectors for top RS universe names (setup candidates)...
  Looking up 60 tickers...


Setup sector/industry lookup:   0%|          | 0/60 [00:00<?, ?it/s]

  Sector map: 117, Industry map: 117

Profiled 596 tickers
Winner profiles: 57
Tickers with sector data: 117


In [8]:
# ===========================================================================
# INDUSTRY GROUP SPONSORSHIP / THRUST SCORE
# ===========================================================================
# For each industry group in the universe, measures breadth of participation:
# - How many names are above 20/50 SMA
# - How many broke out of 20d/50d highs this week
# - What % are Stage 2
# - Median RS and sector-adjusted RS
# - How many are winners this week (group participation in the move)
# This answers: "Is this stock's move isolated, or is the whole group going?"
# ===========================================================================
print("Computing industry group sponsorship scores...\n")

# Only score industries with 3+ names (enough to measure breadth)
MIN_GROUP_SIZE = 3

industry_stats = []
industries_in_universe = features_df[features_df["industry"].notna()]["industry"].unique()

for ind in industries_in_universe:
    group = features_df[features_df["industry"] == ind]
    n = len(group)
    if n < MIN_GROUP_SIZE:
        continue

    pct_above_20  = group["above_sma20"].astype(float).mean()
    pct_above_50  = group["above_sma50"].astype(float).mean()
    pct_breakout_20 = group["breakout_20d"].astype(float).mean()
    pct_breakout_50 = group["breakout_50d"].astype(float).mean()
    pct_stage2    = (group["stage_num"] == 2).mean()
    pct_rs_rising = (group["rs_trend"] == "Rising").mean()
    median_rs     = group["mansfield_rs"].median()
    median_rs_pct = group["rs_percentile"].median()
    median_sect_adj = group["sector_adj_rs"].median()
    median_ud     = group["ud_volume_ratio"].median()
    n_winners     = group["ticker"].isin(winner_tickers).sum()
    pct_winners   = n_winners / n

    # Group Thrust Score (0-100):
    # Breadth (40 pts): above_20 + above_50 + stage2_pct + rs_rising_pct (10 each)
    # Breakout (20 pts): breakout_20d + breakout_50d (10 each)
    # Quality (20 pts): median_rs_pct (10) + median_ud scaled (10)
    # Participation (20 pts): pct of group that are winners this week
    thrust = 0
    thrust += min(10, pct_above_20 * 10)
    thrust += min(10, pct_above_50 * 10)
    thrust += min(10, pct_stage2 * 10)
    thrust += min(10, pct_rs_rising * 10)
    thrust += min(10, pct_breakout_20 * 20)  # breakouts are rare, so scale up
    thrust += min(10, pct_breakout_50 * 20)
    thrust += min(10, (median_rs_pct / 100) * 10) if pd.notna(median_rs_pct) else 0
    thrust += min(10, max(0, (median_ud - 0.5) / 1.5) * 10) if pd.notna(median_ud) else 0
    thrust += min(20, pct_winners * 40)  # scale: 50% winners in group = max 20 pts
    thrust = round(min(100, thrust))

    industry_stats.append({
        "industry": ind,
        "n_names": n,
        "thrust_score": thrust,
        "pct_above_20sma": round(pct_above_20 * 100, 0),
        "pct_above_50sma": round(pct_above_50 * 100, 0),
        "pct_stage2": round(pct_stage2 * 100, 0),
        "pct_rs_rising": round(pct_rs_rising * 100, 0),
        "pct_breakout_20d": round(pct_breakout_20 * 100, 0),
        "pct_breakout_50d": round(pct_breakout_50 * 100, 0),
        "median_rs": round(median_rs, 1) if pd.notna(median_rs) else np.nan,
        "median_rs_pct": round(median_rs_pct, 0) if pd.notna(median_rs_pct) else np.nan,
        "median_sect_adj_rs": round(median_sect_adj, 1) if pd.notna(median_sect_adj) else np.nan,
        "median_ud_vol": round(median_ud, 2) if pd.notna(median_ud) else np.nan,
        "n_winners": int(n_winners),
        "pct_winners": round(pct_winners * 100, 0),
    })

industry_thrust_df = pd.DataFrame(industry_stats).sort_values("thrust_score", ascending=False)

# Assign thrust score to individual stocks in features_df and winner_features
thrust_map = dict(zip(industry_thrust_df["industry"], industry_thrust_df["thrust_score"]))
features_df["group_thrust"] = features_df["industry"].map(thrust_map)
winner_features["group_thrust"] = winner_features["industry"].map(thrust_map)

# Display top groups
print("=" * 80)
print("  INDUSTRY GROUP SPONSORSHIP — TOP 20 BY THRUST SCORE")
print("=" * 80)
print("  (Higher = broader institutional participation across the group)")
print(f"  Minimum group size: {MIN_GROUP_SIZE} names\n")

top_groups = industry_thrust_df.head(20)
display(top_groups.style.format({
    "thrust_score": "{:.0f}",
    "pct_above_20sma": "{:.0f}%",
    "pct_above_50sma": "{:.0f}%",
    "pct_stage2": "{:.0f}%",
    "pct_rs_rising": "{:.0f}%",
    "pct_breakout_20d": "{:.0f}%",
    "pct_breakout_50d": "{:.0f}%",
    "median_rs": "{:+.1f}",
    "median_rs_pct": "{:.0f}",
    "median_sect_adj_rs": "{:+.1f}",
    "median_ud_vol": "{:.2f}",
    "pct_winners": "{:.0f}%",
}).set_caption("Industry Group Thrust Scores"))

# Highlight: which winner tickers are in high-thrust groups?
print(f"\n\nWINNER TICKERS BY GROUP THRUST:")
print("-" * 50)
wf_with_thrust = winner_features[winner_features["group_thrust"].notna()].sort_values(
    ["group_thrust", "breakout_quality"], ascending=[False, False])

for _, r in wf_with_thrust.iterrows():
    if r["tier"] in ["A", "B+", "B"]:
        ind = r.get("industry", "?")
        thrust = r.get("group_thrust", 0)
        label = ""
        if thrust >= 70:
            label = "STRONG GROUP SPONSORSHIP"
        elif thrust >= 50:
            label = "Moderate group support"
        elif thrust >= 30:
            label = "Thin group support"
        else:
            label = "Isolated move"
        print(f"  {r['ticker']:>6s}  Tier:{r['tier']:>2s}  Thrust:{thrust:.0f}  {label}  [{ind}]")

# Bottom groups (weakest — isolated moves or distribution)
print(f"\n\nWEAKEST GROUPS (potential traps):")
print("-" * 50)
bottom = industry_thrust_df.tail(10)
for _, r in bottom.iterrows():
    n_win = r["n_winners"]
    if n_win > 0:
        print(f"  {r['industry']:>40s}: Thrust {r['thrust_score']:.0f}, {r['n_names']} names, {n_win} winners — ISOLATED MOVE WARNING")


Computing industry group sponsorship scores...

  INDUSTRY GROUP SPONSORSHIP — TOP 20 BY THRUST SCORE
  (Higher = broader institutional participation across the group)
  Minimum group size: 3 names



,industry,n_names,thrust_score,pct_above_20sma,pct_above_50sma,pct_stage2,pct_rs_rising,pct_breakout_20d,pct_breakout_50d,median_rs,median_rs_pct,median_sect_adj_rs,median_ud_vol,n_winners,pct_winners
11,Oil & Gas Refining & Marketing,3,89,100%,100%,100%,100%,33%,33%,+22.3,97,+5.6,1.46,3,100%
5,Oil & Gas E&P,8,88,100%,100%,88%,75%,50%,50%,+21.8,96,+5.0,1.57,3,38%
4,Oil & Gas Integrated,13,85,100%,100%,100%,77%,62%,54%,+17.3,94,+0.5,1.81,3,23%
8,Oil & Gas Midstream,7,84,100%,100%,71%,86%,29%,29%,+16.6,93,-0.1,2.02,3,43%
2,Aerospace & Defense,6,70,100%,100%,83%,67%,17%,17%,+17.3,92,+12.1,1.40,2,33%
7,Software - Infrastructure,11,58,100%,45%,0%,100%,36%,0%,-1.6,36,+0.6,0.95,11,100%
0,Software - Application,12,55,100%,17%,0%,100%,42%,0%,-5.4,22,-3.2,0.87,12,100%
12,Telecom Services,3,53,67%,100%,100%,67%,0%,0%,+17.5,94,+14.8,2.01,0,0%
1,Communication Equipment,5,47,60%,80%,100%,0%,0%,0%,+15.1,90,+17.4,1.43,1,20%
6,Scientific & Technical Instruments,3,47,67%,100%,100%,33%,0%,0%,+15.7,91,+17.9,1.66,0,0%




WINNER TICKERS BY GROUP THRUST:
--------------------------------------------------
     MPC  Tier: A  Thrust:89  STRONG GROUP SPONSORSHIP  [Oil & Gas Refining & Marketing]
     PSX  Tier: A  Thrust:89  STRONG GROUP SPONSORSHIP  [Oil & Gas Refining & Marketing]
     VLO  Tier: A  Thrust:89  STRONG GROUP SPONSORSHIP  [Oil & Gas Refining & Marketing]
     WDS  Tier: A  Thrust:88  STRONG GROUP SPONSORSHIP  [Oil & Gas E&P]
     CNQ  Tier: A  Thrust:88  STRONG GROUP SPONSORSHIP  [Oil & Gas E&P]
     EOG  Tier:B+  Thrust:88  STRONG GROUP SPONSORSHIP  [Oil & Gas E&P]
    EQNR  Tier: A  Thrust:85  STRONG GROUP SPONSORSHIP  [Oil & Gas Integrated]
     PBR  Tier: A  Thrust:85  STRONG GROUP SPONSORSHIP  [Oil & Gas Integrated]
      EC  Tier: A  Thrust:85  STRONG GROUP SPONSORSHIP  [Oil & Gas Integrated]
     LNG  Tier:B+  Thrust:84  STRONG GROUP SPONSORSHIP  [Oil & Gas Midstream]
     OKE  Tier: B  Thrust:84  STRONG GROUP SPONSORSHIP  [Oil & Gas Midstream]
    ESLT  Tier: A  Thrust:70  STRONG GR

In [9]:
if MODE == "weekly":
    # ===========================================================================
    # GROUP HEALTH — STRUCTURAL CONDITION PER INDUSTRY
    # ===========================================================================
    # Unlike industry thrust (which measures winner participation), group health
    # measures the ENTIRE industry's condition across the full universe:
    # breadth, momentum, leadership density, cross-sectional strength, quality.
    # ===========================================================================
    print("Computing group health scores...\n")

    MIN_GROUP_HEALTH_SIZE = 3
    group_health_rows = []
    industries = features_df[features_df["industry"].notna()]["industry"].unique()

    # SPY 20d return for relative comparison
    spy_ret_20d_val = (spy_close.iloc[-1] / spy_close.iloc[-21] - 1) if len(spy_close) >= 21 else 0

    for ind in industries:
        g = features_df[features_df["industry"] == ind]
        n = len(g)

        if n < MIN_GROUP_HEALTH_SIZE:
            # Still record but flag as low coverage
            group_health_rows.append({
                "industry": ind, "n_names": n,
                "group_health_score": np.nan, "group_health_label": "LOW_COVERAGE",
                "breadth_score": np.nan, "momentum_score": np.nan,
                "leadership_score": np.nan, "strength_score": np.nan,
                "quality_score": np.nan,
            })
            continue

        # ── Breadth ──
        pct_above_20 = 100 * g["above_sma20"].astype(float).mean()
        pct_above_50 = 100 * g["above_sma50"].astype(float).mean()
        pct_above_200 = 100 * g["above_sma200"].astype(float).mean()
        pct_near_ath = 100 * g["near_ath"].astype(float).mean() if "near_ath" in g.columns else 0
        pct_stage2 = 100 * (g["stage_num"] == 2).mean()

        breadth_score = (
            0.20 * pct_above_20 +
            0.25 * pct_above_50 +
            0.20 * pct_above_200 +
            0.15 * pct_near_ath +
            0.20 * pct_stage2
        )

        # ── Momentum breadth ──
        pct_rising_rs = 100 * (g["rs_trend_code"] == 1).mean()
        pct_macd_bull = 100 * g["macd_bull"].astype(float).mean()
        pct_bo_20 = 100 * g["breakout_20d"].astype(float).mean()
        pct_bo_50 = 100 * g["breakout_50d"].astype(float).mean()

        momentum_score = (
            0.30 * pct_rising_rs +
            0.20 * pct_macd_bull +
            0.30 * pct_bo_20 +
            0.20 * pct_bo_50
        )

        # ── Cross-sectional strength ──
        med_rs_pct = g["rs_percentile"].median()
        med_mansfield = g["mansfield_rs"].median()
        med_sector_adj = g["sector_adj_rs"].median()
        med_ret_20d = g["ret_20d"].median()
        group_vs_spy = (med_ret_20d - spy_ret_20d_val) if pd.notna(med_ret_20d) else 0

        rs_sc = np.clip(50 + 0.5 * (med_rs_pct if pd.notna(med_rs_pct) else 0), 0, 100)
        mans_sc = np.clip(50 + 5 * (med_mansfield if pd.notna(med_mansfield) else 0), 0, 100)
        sadj_sc = np.clip(50 + 3 * (med_sector_adj if pd.notna(med_sector_adj) else 0), 0, 100)
        ret_sc = np.clip(50 + 250 * group_vs_spy, 0, 100)

        strength_score = (rs_sc + mans_sc + sadj_sc + ret_sc) / 4

        # ── Leadership density ──
        leaders_90 = ((g["stage_num"] == 2) & (g["rs_percentile"] >= 90)).sum()
        leaders_80 = ((g["stage_num"] == 2) & (g["rs_percentile"] >= 80)).sum()
        leader_density = leaders_90 / n
        leadership_score = np.clip(100 * (0.7 * leader_density + 0.3 * (leaders_80 / n)), 0, 100)

        # ── Quality / risk ──
        med_wcr = g["weekly_close_range"].median() if "weekly_close_range" in g.columns else 0.5
        med_ud = g["ud_volume_ratio"].median() if "ud_volume_ratio" in g.columns else 1.0
        med_fr = g["failure_risk"].median() if "failure_risk" in g.columns else 3.0
        med_wcr = med_wcr if pd.notna(med_wcr) else 0.5
        med_ud = med_ud if pd.notna(med_ud) else 1.0
        med_fr = med_fr if pd.notna(med_fr) else 3.0

        quality_score = np.clip(
            35 * med_wcr + 20 * min(med_ud, 2.5) - 6 * med_fr,
            0, 100
        )

        # ── Final group health ──
        ghs = (
            0.30 * breadth_score +
            0.20 * momentum_score +
            0.20 * leadership_score +
            0.20 * strength_score +
            0.10 * quality_score
        )
        ghs = round(min(100, max(0, ghs)), 1)

        if ghs >= 80:
            label = "SPONSORED"
        elif ghs >= 65:
            label = "HEALTHY"
        elif ghs >= 50:
            label = "MIXED"
        else:
            label = "WEAK"

        group_health_rows.append({
            "industry": ind,
            "n_names": n,
            "group_health_score": ghs,
            "group_health_label": label,
            "breadth_score": round(breadth_score, 1),
            "momentum_score": round(momentum_score, 1),
            "leadership_score": round(leadership_score, 1),
            "strength_score": round(strength_score, 1),
            "quality_score": round(quality_score, 1),
            # Detail metrics for audit
            "pct_above_20": round(pct_above_20, 0),
            "pct_above_50": round(pct_above_50, 0),
            "pct_above_200": round(pct_above_200, 0),
            "pct_stage2": round(pct_stage2, 0),
            "pct_rising_rs": round(pct_rising_rs, 0),
            "pct_breakout_20d": round(pct_bo_20, 0),
            "pct_breakout_50d": round(pct_bo_50, 0),
            "median_rs_pct": round(med_rs_pct, 0) if pd.notna(med_rs_pct) else np.nan,
            "median_mansfield": round(med_mansfield, 1) if pd.notna(med_mansfield) else np.nan,
            "median_sector_adj": round(med_sector_adj, 1) if pd.notna(med_sector_adj) else np.nan,
            "leader_density": round(leader_density, 2),
            "median_weekly_close": round(med_wcr, 2),
            "median_ud": round(med_ud, 2),
            "median_failure_risk": round(med_fr, 1),
        })

    group_health_df = pd.DataFrame(group_health_rows)
    group_health_df = group_health_df.sort_values("group_health_score", ascending=False, na_position="last")

    # ── Sector-level fallback for LOW_COVERAGE industries ──
    # Compute health at sector level for any industry with < 3 names
    print("\n  Computing sector-level fallback for LOW_COVERAGE industries...")
    sector_health = {}
    for sect in features_df["sector"].dropna().unique():
        sg = features_df[features_df["sector"] == sect]
        if len(sg) < 3:
            continue
        s_pct_above_50 = 100 * sg["above_sma50"].astype(float).mean()
        s_pct_stage2 = 100 * (sg["stage_num"] == 2).mean()
        s_pct_rising = 100 * (sg["rs_trend_code"] == 1).mean()
        s_med_rs = sg["rs_percentile"].median()
        s_med_ud = sg["ud_volume_ratio"].median() if "ud_volume_ratio" in sg.columns else 1.0
        s_med_rs = s_med_rs if pd.notna(s_med_rs) else 50
        s_med_ud = s_med_ud if pd.notna(s_med_ud) else 1.0
        s_health = (0.30 * s_pct_above_50 + 0.25 * s_pct_stage2 +
                    0.25 * s_pct_rising + 0.20 * s_med_rs)
        s_health = round(min(100, max(0, s_health)), 1)
        s_label = "SPONSORED" if s_health >= 80 else "HEALTHY" if s_health >= 65 else "MIXED" if s_health >= 50 else "WEAK"
        sector_health[sect] = (s_health, s_label)
        print(f"    {sect:>25s}: {s_health:.0f} [{s_label}]")

    # Apply fallback: for LOW_COVERAGE industries, use sector-level score
    for idx, row in group_health_df.iterrows():
        if row["group_health_label"] == "LOW_COVERAGE":
            # Find sector for this industry from any stock in features_df
            ind_stocks = features_df[features_df["industry"] == row["industry"]]
            if len(ind_stocks) > 0:
                sect = ind_stocks.iloc[0].get("sector", "")
                if sect in sector_health:
                    fb_score, fb_label = sector_health[sect]
                    group_health_df.at[idx, "group_health_score"] = fb_score
                    group_health_df.at[idx, "group_health_label"] = f"{fb_label}*"  # asterisk = sector fallback
                    print(f"    Fallback: {row['industry']:>35s} -> {sect} ({fb_score:.0f} [{fb_label}*])")

    group_health_df = group_health_df.sort_values("group_health_score", ascending=False, na_position="last")

    # Merge into features_df and winner_features
    health_map = dict(zip(group_health_df["industry"], group_health_df["group_health_score"]))
    health_label_map = dict(zip(group_health_df["industry"], group_health_df["group_health_label"]))

    features_df["group_health_score"] = features_df["industry"].map(health_map)
    features_df["group_health_label"] = features_df["industry"].map(health_label_map)
    winner_features = features_df[features_df["ticker"].isin(winner_tickers)].copy()

    # Display top groups
    scored = group_health_df[group_health_df["group_health_score"].notna()]
    print("=" * 75)
    print("  GROUP HEALTH — TOP 15 INDUSTRIES BY STRUCTURAL CONDITION")
    print("=" * 75)

    gh_display = ["industry", "n_names", "group_health_score", "group_health_label",
                   "breadth_score", "momentum_score", "leadership_score", "strength_score",
                   "quality_score", "pct_stage2", "pct_rising_rs", "leader_density"]

    display(scored.head(15)[gh_display].style.format({
        "group_health_score": "{:.0f}", "breadth_score": "{:.0f}",
        "momentum_score": "{:.0f}", "leadership_score": "{:.0f}",
        "strength_score": "{:.0f}", "quality_score": "{:.0f}",
        "pct_stage2": "{:.0f}%", "pct_rising_rs": "{:.0f}%",
        "leader_density": "{:.0%}",
    }).set_caption("Group Health (higher = structurally stronger across full universe)"))

    # Show bottom groups with winners (potential traps)
    bottom_with_winners = scored[
        (scored["group_health_score"] < 50) &
        (scored["industry"].isin(winner_features["industry"].dropna().unique()))
    ]
    if len(bottom_with_winners) > 0:
        print(f"\n  WEAK GROUPS WITH WINNERS (potential traps):")
        for _, r in bottom_with_winners.iterrows():
            w_in_group = winner_features[winner_features["industry"] == r["industry"]]["ticker"].tolist()
            print(f"    {r['industry']:>40s}  Health:{r['group_health_score']:.0f} [{r['group_health_label']}]  "
                  f"Winners: {', '.join(w_in_group[:5])}")

    n_sponsored = (scored["group_health_label"] == "SPONSORED").sum()
    n_healthy = (scored["group_health_label"] == "HEALTHY").sum()
    n_mixed = (scored["group_health_label"] == "MIXED").sum()
    n_weak = (scored["group_health_label"] == "WEAK").sum()
    print(f"\n  Distribution: {n_sponsored} SPONSORED | {n_healthy} HEALTHY | {n_mixed} MIXED | {n_weak} WEAK")
    print(f"  Scored {len(scored)} industries ({group_health_df['group_health_label'].eq('LOW_COVERAGE').sum()} below minimum size)")


Computing group health scores...


  Computing sector-level fallback for LOW_COVERAGE industries...
                   Technology: 49 [WEAK]
              Basic Materials: 87 [SPONSORED]
       Communication Services: 60 [MIXED]
                  Industrials: 79 [HEALTHY]
            Consumer Cyclical: 45 [WEAK]
                       Energy: 90 [SPONSORED]
           Financial Services: 39 [WEAK]
                  Real Estate: 72 [HEALTHY]
           Consumer Defensive: 82 [SPONSORED]
                   Healthcare: 79 [HEALTHY]
    Fallback:                                Gold -> Basic Materials (87 [SPONSORED*])
    Fallback:                Advertising Agencies -> Communication Services (60 [MIXED*])
    Fallback:                   Metal Fabrication -> Industrials (79 [HEALTHY*])
    Fallback:                     Travel Services -> Consumer Cyclical (45 [WEAK*])
    Fallback:     Information Technology Services -> Technology (49 [WEAK*])
    Fallback:    Financial Data & Stock Exchan

,industry,n_names,group_health_score,group_health_label,breadth_score,momentum_score,leadership_score,strength_score,quality_score,pct_stage2,pct_rising_rs,leader_density
1,Gold,1,87,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
27,Specialty Chemicals,1,87,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
30,Agricultural Inputs,1,87,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
16,Chemicals,1,87,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
28,Oil & Gas Refining & Marketing,3,86,SPONSORED,100,67,100,88,55,100%,100%,100%
26,Grocery Stores,1,82,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
35,Discount Stores,1,82,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
25,Confectioners,1,82,SPONSORED*,nan,nan,nan,nan,nan,nan%,nan%,nan%
8,Oil & Gas Integrated,13,82,SPONSORED,100,62,84,81,65,100%,77%,77%
10,Oil & Gas E&P,8,81,SPONSORED,98,65,79,90,49,88%,75%,75%



  WEAK GROUPS WITH WINNERS (potential traps):
             Information Technology Services  Health:49 [WEAK*]  Winners: BR, IBM
                              Semiconductors  Health:49 [WEAK*]  Winners: MRVL
                                    Gambling  Health:45 [WEAK*]  Winners: FLUT
                             Travel Services  Health:45 [WEAK*]  Winners: BKNG, EXPE
                             Credit Services  Health:39 [WEAK*]  Winners: SOFI
            Financial Data & Stock Exchanges  Health:39 [WEAK*]  Winners: COIN
                   Software - Infrastructure  Health:38 [WEAK]  Winners: CRWD, FTNT, IOT, NET, ORCL
                      Software - Application  Health:32 [WEAK]  Winners: ADBE, ADP, ADSK, DDOG, INTU
                               Entertainment  Health:24 [WEAK]  Winners: FOX, FOXA

  Distribution: 3 SPONSORED | 3 HEALTHY | 4 MIXED | 3 WEAK
  Scored 41 industries (1 below minimum size)


In [10]:
if MODE == "weekly":
    # ===========================================================================
    # PERSISTENCE: LOAD HISTORY + COMPUTE LABELS
    # ===========================================================================
    print("Computing persistence from historical snapshots...\n")

    LOOKBACK_WEEKS = 4

    # -- Load all ticker snapshots for this model version --
    snap_files = sorted([f for f in os.listdir(SNAP_DIR) if f.endswith('.csv')])
    history_frames = []
    for f in snap_files:
        try:
            df = pd.read_csv(os.path.join(SNAP_DIR, f))
            if "model_version" in df.columns and (df["model_version"] == MODEL_VERSION).any():
                history_frames.append(df[df["model_version"] == MODEL_VERSION])
        except Exception:
            pass

    if history_frames:
        history = pd.concat(history_frames, ignore_index=True)
        history["run_date"] = pd.to_datetime(history["run_date"])
        # Deduplicate by ISO week (handles mid-week reruns safely)
        if "iso_week" in history.columns:
            unique_weeks = sorted(history["iso_week"].unique(), reverse=True)[:LOOKBACK_WEEKS]
            # Keep only latest run per week
            history = history.sort_values("run_date", ascending=False).drop_duplicates(
                subset=["ticker", "iso_week"], keep="first")
            history = history[history["iso_week"].isin(unique_weeks)]
            n_weeks = len(unique_weeks)
        else:
            unique_dates = sorted(history["run_date"].unique(), reverse=True)[:LOOKBACK_WEEKS]
            history = history[history["run_date"].isin(unique_dates)]
            n_weeks = len(unique_dates)
        dates_in_history = sorted(history["run_date"].unique())
        print(f"  Loaded {len(history)} rows across {n_weeks} weeks")
        if len(dates_in_history) > 0:
            print(f"  Date range: {pd.Timestamp(min(dates_in_history)).strftime('%Y-%m-%d')} to {pd.Timestamp(max(dates_in_history)).strftime('%Y-%m-%d')}")
    else:
        history = pd.DataFrame()
        n_weeks = 0
        print("  No prior snapshots found — this is the first run.")

    # -- Load group snapshots --
    group_files = sorted([f for f in os.listdir(GROUP_DIR) if f.endswith('.csv')])
    group_history_frames = []
    for f in group_files:
        try:
            df = pd.read_csv(os.path.join(GROUP_DIR, f))
            if "model_version" in df.columns and (df["model_version"] == MODEL_VERSION).any():
                group_history_frames.append(df[df["model_version"] == MODEL_VERSION])
        except Exception:
            pass

    if group_history_frames:
        group_history = pd.concat(group_history_frames, ignore_index=True)
        group_history["run_date"] = pd.to_datetime(group_history["run_date"])
        unique_gdates = sorted(group_history["run_date"].unique(), reverse=True)[:LOOKBACK_WEEKS]
        group_history = group_history[group_history["run_date"].isin(unique_gdates)]
    else:
        group_history = pd.DataFrame()

    # -- Compute ticker persistence --
    def compute_persistence(ticker, tier, history_df, n_weeks):
        """Compute persistence metrics for a single ticker."""
        if history_df.empty or n_weeks == 0:
            return {"persistence_label": "Fresh", "weeks_in_a": 0,
                    "weeks_in_ab": 0, "appearances": 0, "score_trend": np.nan}

        t_hist = history_df[history_df["ticker"] == ticker].sort_values("run_date")
        appearances = t_hist["run_date"].nunique()

        # Consecutive weeks in A-tier (most recent streak)
        weeks_in_a = 0
        for _, row in t_hist.sort_values("run_date", ascending=False).iterrows():
            if row.get("tier") == "A":
                weeks_in_a += 1
            else:
                break

        # Consecutive weeks in A or B+
        weeks_in_ab = 0
        for _, row in t_hist.sort_values("run_date", ascending=False).iterrows():
            if row.get("tier") in ["A", "B+"]:
                weeks_in_ab += 1
            else:
                break

        # Score trend: latest score minus earliest score in window
        if len(t_hist) >= 2 and "breakout_quality" in t_hist.columns:
            scores = t_hist.sort_values("run_date")["breakout_quality"].dropna()
            if len(scores) >= 2:
                score_trend = scores.iloc[-1] - scores.iloc[0]
            else:
                score_trend = np.nan
        else:
            score_trend = np.nan

        # Label
        if tier == "A" and weeks_in_a >= 3:
            label = f"{weeks_in_a}wk Persistent"
        elif tier in ["A", "B+"] and weeks_in_ab >= 2:
            label = f"{weeks_in_ab}wk Persistent"
        elif appearances == 0:
            label = "Fresh"
        elif appearances >= 2 and tier in ["A", "B+", "B"]:
            label = f"{appearances}wk Returning"
        elif appearances >= 2 and tier == "C":
            label = "Fading"
        else:
            label = "Fresh"

        return {"persistence_label": label, "weeks_in_a": weeks_in_a,
                "weeks_in_ab": weeks_in_ab, "appearances": appearances,
                "score_trend": score_trend}

    # Apply to winner_features
    persistence_rows = []
    for _, row in winner_features.iterrows():
        p = compute_persistence(row["ticker"], row["tier"], history, n_weeks)
        p["ticker"] = row["ticker"]
        persistence_rows.append(p)

    if persistence_rows:
        persist_df = pd.DataFrame(persistence_rows)
        winner_features = winner_features.merge(persist_df, on="ticker", how="left")

    # (Setup persistence is computed later after the approaching breakout scanner runs)

    # -- Compute group persistence --
    if not group_history.empty and "industry" in group_history.columns:
        print("\n  Group persistence:")
        for ind in industry_thrust_df["industry"].head(10):
            g_hist = group_history[group_history["industry"] == ind]
            weeks_in_top = g_hist["run_date"].nunique()
            if weeks_in_top > 0:
                print(f"    {ind:>40s}: {weeks_in_top} weeks in history")
    else:
        print("\n  No group history yet.")

    print(f"\nPersistence labels assigned to {len(winner_features)} winners (setup persistence computed later)")
    if n_weeks == 0:
        print("All labels are 'Fresh' — persistence will build after 2+ weekly runs.")


Computing persistence from historical snapshots...

  Loaded 82 rows across 1 weeks
  Date range: 2026-03-08 to 2026-03-08

  Group persistence:
              Oil & Gas Refining & Marketing: 1 weeks in history
                               Oil & Gas E&P: 1 weeks in history
                        Oil & Gas Integrated: 1 weeks in history
                         Oil & Gas Midstream: 1 weeks in history
                         Aerospace & Defense: 1 weeks in history
                   Software - Infrastructure: 1 weeks in history
                      Software - Application: 1 weeks in history
                            Telecom Services: 1 weeks in history
                     Communication Equipment: 1 weeks in history
          Scientific & Technical Instruments: 1 weeks in history

Persistence labels assigned to 57 winners (setup persistence computed later)


In [11]:
if MODE == "weekly":
    # ===========================================================================
    # TECHNICAL COMMONALITIES: WINNERS vs UNIVERSE
    # ===========================================================================

    bool_metrics = [
        "above_sma20", "above_sma50", "above_sma200",
        "breakout_20d", "breakout_50d", "macd_bull"
    ]
    num_metrics = [
        "rsi14", "mansfield_rs", "rs_percentile", "sector_adj_rs",
        "rel_volume_20", "ud_volume_ratio", "breakout_day_vol",
        "atr14_pct", "pct_of_52w_high", "weekly_close_range",
        "ret_5d", "ret_20d",
        "base_depth", "base_tightness", "vol_contraction",
        "breakout_quality"
    ]

    common_rows = []
    for col in bool_metrics:
        w_val = round(100 * winner_features[col].astype(float).mean(), 1) if col in winner_features else np.nan
        u_val = round(100 * features_df[col].astype(float).mean(), 1) if col in features_df else np.nan
        delta = round(w_val - u_val, 1) if pd.notna(w_val) and pd.notna(u_val) else np.nan
        common_rows.append({"metric": col, "winners": w_val, "universe": u_val, "delta": delta, "unit": "%_true"})

    for col in num_metrics:
        w_val = round(winner_features[col].median(), 4) if col in winner_features else np.nan
        u_val = round(features_df[col].median(), 4) if col in features_df else np.nan
        delta = round(w_val - u_val, 4) if pd.notna(w_val) and pd.notna(u_val) else np.nan
        common_rows.append({"metric": col, "winners": w_val, "universe": u_val, "delta": delta, "unit": "median"})

    commonalities = pd.DataFrame(common_rows)

    print("=" * 70)
    print("  TECHNICAL COMMONALITIES: WINNERS vs UNIVERSE")
    print("=" * 70)
    display(commonalities.style.format({
        "winners": "{:.2f}", "universe": "{:.2f}", "delta": "{:+.2f}"
    }).set_caption("Winners vs Universe (delta = winner - universe)"))

    # ===========================================================================
    # FUNDAMENTAL FLAGS + FAILURE RISK
    # ===========================================================================
    if "fund_flag" in winner_features.columns:
        print("\nFUNDAMENTAL FLAGS (winners):")
        for flag in ["FUND_OK", "FUND_MIXED", "FUND_WEAK", "FUND_UNKNOWN"]:
            cnt = (winner_features["fund_flag"] == flag).sum()
            if cnt > 0:
                names = winner_features[winner_features["fund_flag"]==flag]["ticker"].tolist()
                print(f"  {flag:>12s}: {cnt} — {', '.join(names[:10])}")

    if "failure_risk" in winner_features.columns:
        print("\nFAILURE RISK:")
        for label_name in ["Winners", "Universe"]:
            df_tmp = winner_features if label_name == "Winners" else features_df
            avg = df_tmp["failure_risk"].mean()
            lo = (df_tmp["failure_risk"] <= 2).sum()
            hi = (df_tmp["failure_risk"] >= 5).sum()
            print(f"  {label_name:>10s}: avg={avg:.1f}  low-risk(<=2)={lo}  high-risk(>=5)={hi}")

    # ===========================================================================
    # WEINSTEIN STAGE DISTRIBUTION
    # ===========================================================================
    print("\n\nSTAGE DISTRIBUTION:")
    print("-" * 50)
    for label in ["Winners", "Universe"]:
        df_tmp = winner_features if label == "Winners" else features_df
        dist = df_tmp["stage_num"].value_counts().sort_index()
        total = len(df_tmp)
        parts = []
        for s in [1, 2, 3, 4]:
            n = dist.get(s, 0)
            parts.append(f"S{s}: {n} ({100*n/total:.0f}%)")
        print(f"  {label:>10s}: {' | '.join(parts)}")

    # ===========================================================================
    # RS TREND DISTRIBUTION
    # ===========================================================================
    print("\n\nRS TREND DISTRIBUTION:")
    print("-" * 50)
    for label in ["Winners", "Universe"]:
        df_tmp = winner_features if label == "Winners" else features_df
        dist = df_tmp["rs_trend"].value_counts()
        total = df_tmp["rs_trend"].notna().sum()
        parts = []
        for trend in ["Rising", "Flat", "Declining"]:
            n = dist.get(trend, 0)
            pct = 100*n/total if total > 0 else 0
            parts.append(f"{trend}: {n} ({pct:.0f}%)")
        print(f"  {label:>10s}: {' | '.join(parts)}")

    # ===========================================================================
    # SECTOR BREAKDOWN FOR WINNERS
    # ===========================================================================
    print("\n\nWINNER SECTOR BREAKDOWN:")
    print("-" * 50)
    if "sector" in winner_features.columns:
        sector_summary = winner_features.groupby("sector").agg(
            count=("ticker", "count"),
            avg_return=("ret_5d", "mean"),
            avg_rs=("mansfield_rs", "mean"),
            avg_sect_adj_rs=("sector_adj_rs", "mean"),
        ).sort_values("count", ascending=False)
        display(sector_summary.style.format({
            "avg_return": "{:.1%}", "avg_rs": "{:+.1f}", "avg_sect_adj_rs": "{:+.1f}"
        }))

    # ===========================================================================
    # POSITION SIZING FOR A AND B+ TIERS
    # ===========================================================================
    sizing_rows = []
    for _, row in winner_features.iterrows():
        if row.get("tier") in ["A", "B+"]:
            s1 = position_size(row["close"], row["stop_1atr"])
            s2 = position_size(row["close"], row["stop_2atr"])
            sizing_rows.append({
                "ticker": row["ticker"],
                "shares_1atr": s1["shares"], "position_value_1atr": s1["position_value"],
                "pct_of_account_1atr": s1["pct_of_account"], "risk_pct_1atr": s1["actual_risk_pct"],
                "capped_1atr": s1["capped"],
                "shares_2atr": s2["shares"], "position_value_2atr": s2["position_value"],
                "pct_of_account_2atr": s2["pct_of_account"], "risk_pct_2atr": s2["actual_risk_pct"],
                "capped_2atr": s2["capped"],
            })

    if sizing_rows:
        sizing_df = pd.DataFrame(sizing_rows)
        winner_features = winner_features.merge(sizing_df, on="ticker", how="left")

    # ===========================================================================
    # TIERED WATCHLIST
    # ===========================================================================
    display_cols = [
        "tier", "master_rank", "ticker", "persistence_label", "sector", "industry",
        "group_thrust", "group_health_score", "group_health_label",
        "fund_flag", "failure_risk", "weekly_return", "close",
        "stage_label", "rs_percentile", "mansfield_rs", "rs_trend", "sector_adj_rs",
        "breakout_quality", "near_ath", "weekly_close_range",
        "pct_to_pivot_20d", "breakout_day_vol", "rel_volume_20", "ud_volume_ratio",
        "pct_of_52w_high", "rsi14", "atr14_pct",
        "base_depth", "vol_contraction",
        "stop_1atr", "shares_1atr", "position_value_1atr", "risk_pct_1atr",
        "stop_2atr", "shares_2atr", "position_value_2atr", "risk_pct_2atr"
    ]
    display_cols = [c for c in display_cols if c in winner_features.columns]

    # Explicit tier ordering (lexicographic would put B before B+)
    tier_rank = {"A": 0, "B+": 1, "B": 2, "C": 3}
    winner_features["_tier_rank"] = winner_features["tier"].map(tier_rank).fillna(9)
    wf_sorted = winner_features.sort_values(
        ["_tier_rank", "breakout_quality", "group_thrust", "rs_percentile"],
        ascending=[True, False, False, False]
    ).drop(columns=["_tier_rank"])

    # Master rank: single column for consistent ordering across display, report, and CSV
    wf_sorted = wf_sorted.reset_index(drop=True)
    wf_sorted["master_rank"] = range(1, len(wf_sorted) + 1)
    winner_features["master_rank"] = winner_features["ticker"].map(
        dict(zip(wf_sorted["ticker"], wf_sorted["master_rank"])))

    format_dict = {
        "weekly_return": "{:.1%}",
        "close": "${:.2f}",
        "rs_percentile": "{:.0f}",
        "mansfield_rs": "{:+.1f}",
        "sector_adj_rs": "{:+.1f}",
        "breakout_quality": "{:.0f}",
        "group_thrust": "{:.0f}",
    "group_health_score": "{:.0f}",
        "failure_risk": "{:.0f}",
        "master_rank": "{:.0f}",
        "rev_growth": "{:.0%}",
        "eps_growth": "{:.0%}",
        "weekly_close_range": "{:.0%}",
        "pct_to_pivot_20d": "{:+.1%}",
        "breakout_day_vol": "{:.1f}x",
        "rel_volume_20": "{:.2f}x",
        "ud_volume_ratio": "{:.2f}",
        "pct_of_52w_high": "{:.1%}",
        "rsi14": "{:.0f}",
        "atr14_pct": "{:.1%}",
        "base_depth": "{:.1%}",
        "vol_contraction": "{:.2f}",
        "stop_1atr": "${:.2f}",
        "stop_2atr": "${:.2f}",
        "shares_1atr": "{:.0f}",
        "position_value_1atr": "${:,.0f}",
        "risk_pct_1atr": "{:.2%}",
        "shares_2atr": "{:.0f}",
        "position_value_2atr": "${:,.0f}",
        "risk_pct_2atr": "{:.2%}",
    }

    tier_descriptions = [
        ("A",  "ACTIONABLE - Stage 2 breakouts, strong RS rising, volume confirmed, near ATH"),
        ("B+", "STAGE 3 WATCH - Near highs with strong RS, SMA flattening, watch for re-entry"),
        ("B",  "WATCHLIST - Stage 1-2 with potential, needs confirmation"),
        ("C",  "AVOID - Stage 3/4, dead cat bounces, declining RS"),
    ]

    for tier_name, tier_desc in tier_descriptions:
        subset = wf_sorted[wf_sorted["tier"] == tier_name]
        print(f"\n{'='*70}")
        print(f"  TIER {tier_name}: {tier_desc}  ({len(subset)} names)")
        print(f"{'='*70}")
        if len(subset) > 0:
            display(subset[display_cols].style.format(
                {k: v for k, v in format_dict.items() if k in display_cols}
            ))
        else:
            print("  (none this week)")

    print(f"\n{'='*70}")
    print(f"  POSITION SIZING (${ACCOUNT_SIZE:,.0f} account, {RISK_PER_TRADE:.0%} risk/trade, {MAX_POSITION_PCT:.0%} max position)")
    print(f"{'='*70}")
    print(f"  Dollar risk per trade: ${ACCOUNT_SIZE * RISK_PER_TRADE:,.0f}")
    print(f"  Max position value: ${ACCOUNT_SIZE * MAX_POSITION_PCT:,.0f}")
    print(f"  1-ATR stop = tighter stop, more shares. 2-ATR stop = wider stop, fewer shares.")


  TECHNICAL COMMONALITIES: WINNERS vs UNIVERSE


,metric,winners,universe,delta,unit
0,above_sma20,96.50,34.10,+62.40,%_true
1,above_sma50,49.10,46.10,+3.00,%_true
2,above_sma200,36.80,63.60,-26.80,%_true
3,breakout_20d,43.90,7.70,+36.20,%_true
4,breakout_50d,19.30,4.70,+14.60,%_true
5,macd_bull,96.50,29.50,+67.00,%_true
6,rsi14,60.13,46.01,+14.12,median
7,mansfield_rs,-0.64,2.52,-3.15,median
8,rs_percentile,40.40,49.90,-9.50,median
9,sector_adj_rs,0.62,5.55,-4.94,median



FUNDAMENTAL FLAGS (winners):
       FUND_OK: 22 — ADBE, ADP, ADSK, APP, BKNG, BR, CNQ, ESLT, IBM, KR
    FUND_MIXED: 27 — ASTS, AXON, CRWD, CSGP, DDOG, EXPE, FLUT, FOX, FOXA, FTNT
     FUND_WEAK: 5 — EC, EOG, EQNR, TGT, WDS
  FUND_UNKNOWN: 3 — COIN, DOW, LYB

FAILURE RISK:
     Winners: avg=2.6  low-risk(<=2)=26  high-risk(>=5)=6
    Universe: avg=2.3  low-risk(<=2)=349  high-risk(>=5)=30


STAGE DISTRIBUTION:
--------------------------------------------------
     Winners: S1: 4 (7%) | S2: 15 (26%) | S3: 7 (12%) | S4: 31 (54%)
    Universe: S1: 110 (18%) | S2: 317 (53%) | S3: 46 (8%) | S4: 123 (21%)


RS TREND DISTRIBUTION:
--------------------------------------------------
     Winners: Rising: 50 (88%) | Flat: 7 (12%) | Declining: 0 (0%)
    Universe: Rising: 196 (33%) | Flat: 119 (20%) | Declining: 281 (47%)


WINNER SECTOR BREAKDOWN:
--------------------------------------------------


,count,avg_return,avg_rs,avg_sect_adj_rs
sector,,,,
Technology,27,10.0%,-3.9,-1.7
Energy,12,10.2%,+26.5,+9.8
Communication Services,4,9.4%,-4.2,-6.8
Industrials,3,14.4%,+17.1,+12.0
Consumer Cyclical,3,9.8%,-13.9,-11.5
Basic Materials,2,13.2%,+30.7,+26.6
Financial Services,2,9.3%,-13.0,-9.3
Consumer Defensive,2,7.4%,+17.0,+11.1
Healthcare,1,7.4%,-4.6,-4.7



  TIER A: ACTIONABLE - Stage 2 breakouts, strong RS rising, volume confirmed, near ATH  (10 names)


...


  TIER B: WATCHLIST - Stage 1-2 with potential, needs confirmation  (5 names)


,tier,ticker,persistence_label,sector,industry,group_thrust,group_health_score,group_health_label,fund_flag,failure_risk,close,stage_label,rs_percentile,mansfield_rs,rs_trend,sector_adj_rs,breakout_quality,near_ath,weekly_close_range,pct_to_pivot_20d,breakout_day_vol,rel_volume_20,ud_volume_ratio,pct_of_52w_high,rsi14,atr14_pct,base_depth,vol_contraction,stop_1atr,shares_1atr,position_value_1atr,risk_pct_1atr,stop_2atr,shares_2atr,position_value_2atr,risk_pct_2atr
14,B,MRVL,Fresh,Technology,Semiconductors,nan,49,WEAK*,FUND_OK,0,$89.57,Stage 2 - Advancing,78,+10.2,Rising,+12.4,9,False,79%,+8.8%,5.8x,5.05x,0.97,89.5%,64,4.8%,18.2%,2.10,$85.30,nan,$nan,nan%,$81.04,nan,$nan,nan%
15,B,OKE,Fresh,Energy,Oil & Gas Midstream,84,69,HEALTHY,FUND_MIXED,1,$86.93,Stage 2 - Advancing,88,+13.8,Flat,-3.0,7,False,95%,-0.5%,1.3x,1.09x,1.34,88.4%,62,3.2%,19.0%,1.44,$84.11,nan,$nan,nan%,$81.29,nan,$nan,nan%
16,B,DOW,Fresh,Basic Materials,Chemicals,nan,87,SPONSORED*,FUND_UNKNOWN,0,$33.28,Stage 2 - Advancing,98,+24.4,Flat,+20.3,7,False,68%,-1.3%,1.9x,1.51x,1.24,94.5%,65,4.4%,32.9%,1.65,$31.80,nan,$nan,nan%,$30.32,nan,$nan,nan%
17,B,EXPE,Fresh,Consumer Cyclical,Travel Services,nan,45,WEAK*,FUND_MIXED,1,$249.62,Stage 2 - Advancing,37,-1.4,Rising,+1.0,7,False,95%,-0.8%,2.9x,0.88x,0.69,83.0%,59,5.3%,37.4%,3.72,$236.45,nan,$nan,nan%,$223.27,nan,$nan,nan%
18,B,ASTS,Fresh,Technology,Communication Equipment,47,58,MIXED,FUND_MIXED,1,$89.47,Stage 2 - Advancing,45,+1.4,Flat,+3.7,4,False,40%,-14.7%,1.3x,0.97x,1.17,73.3%,49,9.6%,41.5%,0.82,$80.86,nan,$nan,nan%,$72.24,nan,$nan,nan%



  TIER C: AVOID - Stage 3/4, dead cat bounces, declining RS  (38 names)


,tier,ticker,persistence_label,sector,industry,group_thrust,group_health_score,group_health_label,fund_flag,failure_risk,close,stage_label,rs_percentile,mansfield_rs,rs_trend,sector_adj_rs,breakout_quality,near_ath,weekly_close_range,pct_to_pivot_20d,breakout_day_vol,rel_volume_20,ud_volume_ratio,pct_of_52w_high,rsi14,atr14_pct,base_depth,vol_contraction,stop_1atr,shares_1atr,position_value_1atr,risk_pct_1atr,stop_2atr,shares_2atr,position_value_2atr,risk_pct_2atr
19,C,VG,Fresh,Energy,Oil & Gas Midstream,84,69,HEALTHY,FUND_OK,3,$12.48,Stage 3 - Topping (SMA declining),100,+51.3,Rising,+34.5,9,False,68%,+1.6%,3.5x,1.71x,2.12,65.9%,73,7.3%,45.4%,10.07,$11.57,nan,$nan,nan%,$10.66,nan,$nan,nan%
20,C,IOT,Fresh,Technology,Software - Infrastructure,58,38,WEAK,FUND_MIXED,2,$35.36,Stage 4 - Declining,78,+10.0,Rising,+12.3,7,False,97%,+19.5%,4.0x,3.40x,0.95,74.1%,71,5.5%,34.7%,2.64,$33.40,nan,$nan,nan%,$31.44,nan,$nan,nan%
21,C,TWLO,Fresh,Technology,Software - Infrastructure,58,38,WEAK,FUND_MIXED,3,$128.03,Stage 3 - Topping (SMA declining),54,+3.3,Rising,+5.5,6,False,96%,+2.0%,0.9x,0.56x,1.04,88.8%,61,4.6%,24.7%,2.53,$122.09,nan,$nan,nan%,$116.15,nan,$nan,nan%
22,C,PLTR,Fresh,Technology,Software - Infrastructure,58,38,WEAK,FUND_OK,2,$157.16,Stage 4 - Declining,36,-1.6,Rising,+0.6,6,False,82%,+2.6%,1.5x,1.31x,1.00,75.9%,61,4.5%,33.6%,1.21,$150.16,nan,$nan,nan%,$143.16,nan,$nan,nan%
23,C,SPOT,Fresh,Communication Services,Internet Content & Information,nan,60,MIXED*,FUND_OK,4,$565.19,Stage 4 - Declining,76,+9.7,Rising,+7.0,6,False,96%,+2.4%,1.1x,0.69x,0.89,72.8%,67,4.2%,30.4%,5.03,$541.70,nan,$nan,nan%,$518.20,nan,$nan,nan%
24,C,NET,Fresh,Technology,Software - Infrastructure,58,38,WEAK,FUND_MIXED,3,$195.19,Stage 4 - Declining,64,+5.8,Rising,+8.0,5,False,96%,-0.3%,0.8x,0.51x,1.14,77.1%,58,5.6%,22.2%,4.46,$184.32,nan,$nan,nan%,$173.45,nan,$nan,nan%
25,C,VRSN,Fresh,Technology,Software - Infrastructure,58,38,WEAK,FUND_OK,2,$243.78,Stage 4 - Declining,58,+4.5,Rising,+6.8,5,False,99%,+0.9%,1.3x,0.60x,0.96,80.4%,65,2.8%,16.3%,4.70,$237.06,nan,$nan,nan%,$230.35,nan,$nan,nan%
26,C,CRWD,Fresh,Technology,Software - Infrastructure,58,38,WEAK,FUND_MIXED,3,$428.99,Stage 4 - Declining,33,-2.0,Rising,+0.2,5,False,93%,-0.2%,2.2x,0.85x,0.88,76.9%,56,5.7%,27.2%,2.71,$404.74,nan,$nan,nan%,$380.48,nan,$nan,nan%
27,C,ADSK,Fresh,Technology,Software - Application,55,32,WEAK,FUND_OK,2,$260.99,Stage 4 - Declining,41,-0.3,Rising,+1.9,5,False,83%,-1.2%,1.2x,0.77x,1.28,79.9%,60,3.2%,27.4%,3.45,$252.73,nan,$nan,nan%,$244.47,nan,$nan,nan%
28,C,PAYX,Fresh,Technology,Software - Application,55,32,WEAK,FUND_MIXED,3,$100.85,Stage 4 - Declining,37,-1.4,Rising,+0.9,5,False,98%,+1.8%,1.4x,1.14x,0.78,64.9%,59,3.0%,22.9%,2.00,$97.87,nan,$nan,nan%,$94.88,nan,$nan,nan%



  POSITION SIZING ($100,000 account, 1% risk/trade, 20% max position)
  Dollar risk per trade: $1,000
  Max position value: $20,000
  1-ATR stop = tighter stop, more shares. 2-ATR stop = wider stop, fewer shares.


In [12]:
if MODE == "weekly":
    # ===========================================================================
    # APPROACHING BREAKOUT SCANNER
    # ===========================================================================
    # Scans the FULL universe for names that haven't moved 5%+ yet but are
    # setting up: Stage 2, strong RS, near their pivot, with tightening bases.
    # These are Monday morning watchlist candidates for the coming week.
    # ===========================================================================
    print("Scanning full universe for approaching breakouts...\n")

    # Exclude names already in winners (they already moved)
    non_winners = features_df[~features_df["ticker"].isin(winner_tickers)].copy()

    # SETUP CRITERIA:
    # 1. Stage 2 (advancing)
    # 2. RS percentile > 60
    # 3. RS trend not declining
    # 4. Within 10% of 52-week high (not extended, but close)
    # 5. Within 3% of 20-day high (approaching pivot)
    # 6. Weekly closing range >= 40% (demand present)
    # 7. Up/down volume ratio > 0.9 (not distributing heavily)

    # Compute "distance to 20d pivot" for non-winners
    # Setup criteria: Stage 2, strong RS, and NEAR THE ACTUAL PIVOT
    # pct_to_pivot_20d: 0 = at pivot, -0.03 = 3% below pivot, +0.01 = 1% above
    setup_candidates = non_winners[
        (non_winners["stage_num"] == 2) &
        (non_winners["rs_percentile"] > 60) &
        (non_winners["rs_trend_code"] >= 0) &              # RS not declining
        (non_winners["pct_of_52w_high"] >= 0.85) &         # not in deep drawdown
        (non_winners["pct_of_52w_high"] <= 1.02) &         # not extended > 2% past high
        (non_winners["pct_to_pivot_20d"] >= -0.03) &       # within 3% BELOW the 20d pivot
        (non_winners["pct_to_pivot_20d"] <= 0.005) &       # not already broken out > 0.5% past pivot
        (non_winners["weekly_close_range"] >= 0.40) &      # demand into close
        (non_winners["ud_volume_ratio"] > 0.9) &            # not distributing
        ((non_winners["group_health_score"] >= 55) | (non_winners["group_health_score"].isna()))  # healthy group
    ].copy()

    # Sort by breakout quality, then RS
    setup_candidates = setup_candidates.sort_values(
        ["breakout_quality", "rs_percentile"],
        ascending=[False, False]
    ).head(25)  # Cap at top 25

    # Fill in any missing sector data for setup candidates
    setup_missing_sector = setup_candidates[setup_candidates["sector"].isna()]["ticker"].tolist()
    if setup_missing_sector:
        print(f"Looking up sectors for {len(setup_missing_sector)} setup candidates...")
        for t in tqdm(setup_missing_sector, desc="Setup sector lookup"):
            try:
                tk = yf.Ticker(t)
                info = tk.get_info() if hasattr(tk, 'get_info') else (tk.info or {})
                sect = info.get("sector", "")
                if sect:
                    sector_map[t] = sect
                    setup_candidates.loc[setup_candidates["ticker"] == t, "sector"] = sect
                    ind = info.get("industry", "")
                    if ind:
                        industry_map[t] = ind
                        setup_candidates.loc[setup_candidates["ticker"] == t, "industry"] = ind
                        setup_candidates.loc[setup_candidates["ticker"] == t, "group_thrust"] = thrust_map.get(ind, np.nan)
                    # Compute sector-adjusted RS
                    etf = SECTOR_ETF_MAP.get(sect, "")
                    etf_rs = sector_etf_rs_now.get(etf, np.nan)
                    stock_rs = all_rs_values.get(t, np.nan)
                    setup_candidates.loc[setup_candidates["ticker"] == t, "sector_adj_rs"] = (
                        sector_adjusted_rs(stock_rs, etf_rs))
                time.sleep(0.15)
            except Exception:
                pass
        filled = setup_candidates["sector"].notna().sum()
        print(f"  Setups with sector data: {filled}/{len(setup_candidates)}")

    # Merge group health into setups
if "group_health_score" not in setup_candidates.columns and "health_map" in dir():
    setup_candidates["group_health_score"] = setup_candidates["industry"].map(health_map)
    setup_candidates["group_health_label"] = setup_candidates["industry"].map(health_label_map)

# Setup persistence labels
    setup_persist_rows = []
    for _, row in setup_candidates.iterrows():
        t_hist = history[history["ticker"] == row["ticker"]] if not history.empty else pd.DataFrame()
        appearances = t_hist["run_date"].nunique() if not t_hist.empty else 0
        was_a = (t_hist["tier"] == "A").any() if not t_hist.empty else False
        was_ab = (t_hist["tier"].isin(["A", "B+"])).any() if not t_hist.empty else False

        if was_a:
            label = "Former A-tier"
        elif was_ab:
            label = "Former B+"
        elif appearances >= 2:
            label = f"{appearances}wk Setup"
        else:
            label = "New Setup"

        setup_persist_rows.append({"ticker": row["ticker"], "persistence_label": label,
                                    "appearances": appearances})

    if setup_persist_rows:
        setup_persist_df = pd.DataFrame(setup_persist_rows)
        setup_candidates = setup_candidates.merge(setup_persist_df, on="ticker", how="left")

    # Fund flags + failure risk for setup candidates
    setup_fund_needed = [t for t in setup_candidates["ticker"].tolist() if t not in fund_flags]
    if setup_fund_needed:
        print(f"Fetching fund flags for {len(setup_fund_needed)} setup candidates...")
        for t in tqdm(setup_fund_needed, desc="Setup fundamentals"):
            try:
                tk = yf.Ticker(t)
                info = {}
                try:
                    info = tk.get_info() or {}
                except Exception:
                    try:
                        info = tk.info or {}
                    except Exception:
                        pass
                rev_g = info.get("revenueGrowth")
                eps_g = info.get("earningsGrowth")
                rev_ok = pd.notna(rev_g) and rev_g > 0
                rev_bad = pd.notna(rev_g) and rev_g <= 0
                eps_ok = pd.notna(eps_g) and eps_g > 0
                eps_bad = pd.notna(eps_g) and eps_g <= 0
                if rev_ok and eps_ok: flag = "FUND_OK"
                elif rev_bad and eps_bad: flag = "FUND_WEAK"
                elif (rev_ok or eps_ok) and not (rev_bad and eps_bad): flag = "FUND_MIXED"
                else: flag = "FUND_UNKNOWN"
                fund_flags[t] = {"fund_flag": flag,
                                 "rev_growth": round(rev_g, 3) if pd.notna(rev_g) else np.nan,
                                 "eps_growth": round(eps_g, 3) if pd.notna(eps_g) else np.nan}
                time.sleep(0.15)
            except Exception:
                fund_flags[t] = {"fund_flag": "FUND_UNKNOWN", "rev_growth": np.nan, "eps_growth": np.nan}

    setup_candidates["fund_flag"] = setup_candidates["ticker"].map(
        lambda t: fund_flags.get(t, {}).get("fund_flag", np.nan))
    setup_candidates["failure_risk"] = setup_candidates.apply(failure_risk_score, axis=1)

    # Add position sizing for setups too
    setup_sizing = []
    for _, row in setup_candidates.iterrows():
        s1 = position_size(row["close"], row["stop_1atr"])
        s2 = position_size(row["close"], row["stop_2atr"])
        setup_sizing.append({
            "ticker": row["ticker"],
            "shares_1atr": s1["shares"], "position_value_1atr": s1["position_value"],
            "shares_2atr": s2["shares"], "position_value_2atr": s2["position_value"],
        })

    if setup_sizing:
        setup_sizing_df = pd.DataFrame(setup_sizing)
        setup_candidates = setup_candidates.merge(setup_sizing_df, on="ticker", how="left")

    setup_display_cols = [
        "ticker", "persistence_label", "sector", "fund_flag", "failure_risk", "close", "pct_to_pivot_20d",
        "stage_label", "rs_percentile", "mansfield_rs", "rs_trend", "sector_adj_rs",
        "breakout_quality",
        "pct_of_52w_high", "weekly_close_range",
        "breakout_day_vol", "ud_volume_ratio",
        "base_depth", "vol_contraction",
        "rsi14", "atr14_pct",
        "stop_1atr", "shares_1atr", "position_value_1atr",
        "stop_2atr", "shares_2atr", "position_value_2atr",
    ]
    setup_display_cols = [c for c in setup_display_cols if c in setup_candidates.columns]

    setup_format = {
        "close": "${:.2f}",
        "rs_percentile": "{:.0f}",
        "mansfield_rs": "{:+.1f}",
        "sector_adj_rs": "{:+.1f}",
        "breakout_quality": "{:.0f}",
        "pct_of_52w_high": "{:.1%}",
        "weekly_close_range": "{:.0%}",
        "breakout_day_vol": "{:.1f}x",
        "pct_to_pivot_20d": "{:+.1%}",
        "ud_volume_ratio": "{:.2f}",
        "base_depth": "{:.1%}",
        "vol_contraction": "{:.2f}",
        "rsi14": "{:.0f}",
        "atr14_pct": "{:.1%}",
        "stop_1atr": "${:.2f}",
        "shares_1atr": "{:.0f}",
        "position_value_1atr": "${:,.0f}",
        "stop_2atr": "${:.2f}",
        "shares_2atr": "{:.0f}",
        "position_value_2atr": "${:,.0f}",
    }

    print(f"{'='*70}")
    print(f"  APPROACHING BREAKOUTS — MONDAY MORNING WATCHLIST ({len(setup_candidates)} setups)")
    print(f"{'='*70}")
    print(f"  Stage 2 + RS > 60 rising + within 3% of 20d pivot + not broken out + demand present")
    print(f"  These have NOT yet moved 5%+ — they are APPROACHING their pivots.\n")

    if len(setup_candidates) > 0:
        display(setup_candidates[setup_display_cols].style.format(
            {k: v for k, v in setup_format.items() if k in setup_display_cols}
        ))

        # Sector breakdown of setups
        print(f"\n  Setup sectors:")
        if "sector" in setup_candidates.columns:
            sect_counts = setup_candidates["sector"].value_counts()
            for sect, cnt in sect_counts.items():
                print(f"    {sect}: {cnt}")
    else:
        print("  No approaching breakout setups found this week.")
        print("  (This can happen when the market is in a broad correction.)")


Scanning full universe for approaching breakouts...

Looking up sectors for 14 setup candidates...


Setup sector lookup:   0%|          | 0/14 [00:00<?, ?it/s]

  Setups with sector data: 25/25


In [13]:
if MODE == "weekly":
    # ===========================================================================
    # PRIORITY REVIEW LIST — TOP N NAMES FOR ACTION
    # ===========================================================================
    # Combines Tier A active leaders + top approaching setups into one
    # ranked list with concentration control and regime adjustment.
    # ===========================================================================
    print("\nBuilding Priority Review List...\n")

    # Candidate pool: A-tier winners + top setups
    # (B+ included only if persistence is strong)
    pool_parts = []

    # A-tier winners
    a_tier = winner_features[winner_features["tier"] == "A"].copy()
    a_tier["_status"] = "ACTIVE_LEADER"
    pool_parts.append(a_tier)

    # B+ with persistence (2wk+ or strong group thrust)
    bplus_tier = winner_features[winner_features["tier"] == "B+"].copy()
    if len(bplus_tier) > 0:
        strong_bplus = bplus_tier[
            (bplus_tier.get("persistence_label", pd.Series(dtype=str)).str.contains("Persistent", na=False)) |
            (bplus_tier.get("group_thrust", pd.Series(dtype=float)).fillna(0) >= 70)
        ]
        if len(strong_bplus) > 0:
            strong_bplus["_status"] = "ACTIVE_LEADER"
            pool_parts.append(strong_bplus)

    # Top setups (already filtered to near-pivot)
    if "setup_candidates" in dir() and len(setup_candidates) > 0:
        top_setups = setup_candidates.head(15).copy()
        top_setups["_status"] = "NEW_SETUP"
        # Setups don't have tier column from winners, add placeholder
        if "tier" not in top_setups.columns:
            top_setups["tier"] = "Setup"
        pool_parts.append(top_setups)

    if pool_parts:
        pool = pd.concat(pool_parts, ignore_index=True)

        # Ensure fund_flag is present for setup candidates in the pool
        for idx, row in pool.iterrows():
            if row.get("_status") == "NEW_SETUP":
                t = row["ticker"]
                if pd.isna(row.get("fund_flag")) and t in fund_flags:
                    pool.at[idx, "fund_flag"] = fund_flags[t].get("fund_flag", np.nan)
                    pool.at[idx, "rev_growth"] = fund_flags[t].get("rev_growth", np.nan)
                    pool.at[idx, "eps_growth"] = fund_flags[t].get("eps_growth", np.nan)

        # Compute sizing for any names missing it (setups don't get sized until here)
        for idx, row in pool.iterrows():
            if pd.isna(row.get("shares_1atr")) and pd.notna(row.get("close")) and pd.notna(row.get("stop_1atr")):
                s1 = position_size(row["close"], row["stop_1atr"])
                s2 = position_size(row["close"], row["stop_2atr"])
                pool.at[idx, "shares_1atr"] = s1["shares"]
                pool.at[idx, "position_value_1atr"] = s1["position_value"]
                pool.at[idx, "shares_2atr"] = s2["shares"]
                pool.at[idx, "position_value_2atr"] = s2["position_value"]

        # Greedy portfolio-style selection with iterative concentration penalty
        regime_verdict = regime.get("REGIME_VERDICT", "MIXED / TRANSITIONAL")

        # Step 1: Score everyone with zero concentration penalty
        pool["_base_score"] = pool.apply(
            lambda row: priority_score(row, regime_verdict, concentration_penalty=0), axis=1)

        # Step 2: Greedy portfolio selection with audit trail
        selected = []
        selected_sectors = {}
        remaining = pool.copy()

        for pick_num in range(min(PRIORITY_TOP_N * 2, len(pool))):
            if remaining.empty:
                break

            # Recalculate with current concentration penalties
            penalties = {}
            for idx, row in remaining.iterrows():
                sect = row.get("sector", "")
                count = selected_sectors.get(sect, 0)
                pen = {0: 0, 1: 1.5, 2: 3.0}.get(count, 5.0)
                penalties[idx] = pen

            remaining["_conc_pen"] = remaining.index.map(penalties)
            remaining["priority_score"] = remaining.apply(
                lambda row: priority_score(row, regime_verdict,
                    concentration_penalty=row["_conc_pen"]), axis=1)

            best_idx = remaining["priority_score"].idxmax()
            best = remaining.loc[best_idx]

            selected.append({
                "idx": best_idx,
                "greedy_pick_order": pick_num + 1,
                "raw_priority_score": float(pool.loc[best_idx, "_base_score"]),
                "concentration_penalty": float(penalties[best_idx]),
                "priority_score": float(best["priority_score"]),
            })

            sect = best.get("sector", "")
            if isinstance(sect, str) and sect:
                selected_sectors[sect] = selected_sectors.get(sect, 0) + 1
            remaining = remaining.drop(best_idx)

        # Build final list in greedy pick order
        sel_df = pd.DataFrame(selected)
        pool = pool.loc[sel_df["idx"].tolist()].copy()
        pool["greedy_pick_order"] = sel_df["greedy_pick_order"].values
        pool["raw_priority_score"] = sel_df["raw_priority_score"].values
        pool["concentration_penalty"] = sel_df["concentration_penalty"].values
        pool["priority_score"] = sel_df["priority_score"].values
        pool["priority_rank"] = pool["greedy_pick_order"]

        # Display top N
        top_n = pool.head(PRIORITY_TOP_N)

        # Separate active leaders from setups for display
        active = top_n[top_n["_status"] == "ACTIVE_LEADER"]
        setups_in_top = top_n[top_n["_status"] == "NEW_SETUP"]

        print("=" * 70)
        print(f"  PRIORITY REVIEW — TOP {PRIORITY_TOP_N} NAMES FOR ATTENTION")
        print(f"  Regime: {regime_verdict}")
        print("=" * 70)

        priority_cols = ["priority_rank", "ticker", "_status",
                         "priority_score", "raw_priority_score", "concentration_penalty",
                         "group_health_score",
                         "sector", "industry", "group_thrust", "breakout_quality",
                         "failure_risk", "fund_flag", "rs_percentile", "sector_adj_rs",
                         "weekly_close_range", "ud_volume_ratio", "pct_of_52w_high",
                         "close", "stop_1atr", "shares_1atr", "stop_2atr", "shares_2atr"]
        priority_cols = [c for c in priority_cols if c in pool.columns]

        priority_fmt = {
            "priority_score": "{:.1f}", "raw_priority_score": "{:.1f}",
            "group_health_score": "{:.0f}",
            "concentration_penalty": "{:.1f}", "breakout_quality": "{:.0f}",
            "failure_risk": "{:.0f}", "rs_percentile": "{:.0f}",
            "sector_adj_rs": "{:+.1f}", "group_thrust": "{:.0f}",
            "weekly_close_range": "{:.0%}", "ud_volume_ratio": "{:.2f}",
            "pct_of_52w_high": "{:.1%}", "close": "${:.2f}",
            "stop_1atr": "${:.2f}", "shares_1atr": "{:.0f}",
            "stop_2atr": "${:.2f}", "shares_2atr": "{:.0f}",
        }

        if len(active) > 0:
            print(f"\n  ACTIVE LEADERS ({len(active)}):")
            display(active[priority_cols].style.format(
                {k: v for k, v in priority_fmt.items() if k in priority_cols}))

        if len(setups_in_top) > 0:
            print(f"\n  NEW SETUPS — WAIT FOR TRIGGER ({len(setups_in_top)}):")
            display(setups_in_top[priority_cols].style.format(
                {k: v for k, v in priority_fmt.items() if k in priority_cols}))

        # Save for export
        priority_pool = pool
        print(f"\n  Full priority pool: {len(pool)} candidates scored")
    else:
        print("  No candidates for priority list.")
        priority_pool = pd.DataFrame()



Building Priority Review List...

  PRIORITY REVIEW — TOP 8 NAMES FOR ATTENTION
  Regime: MIXED / TRANSITIONAL

  ACTIVE LEADERS (6):


/tmp/ipykernel_486/1812290086.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  strong_bplus["_status"] = "ACTIVE_LEADER"


,priority_rank,ticker,_status,priority_score,raw_priority_score,concentration_penalty,group_health_score,sector,industry,group_thrust,breakout_quality,failure_risk,fund_flag,rs_percentile,sector_adj_rs,weekly_close_range,ud_volume_ratio,pct_of_52w_high,close,stop_1atr,shares_1atr,stop_2atr,shares_2atr
3,1,ESLT,ACTIVE_LEADER,56.6,56.6,0.0,72,Industrials,Aerospace & Defense,70,13,2,FUND_OK,100,+38.6,85%,2.18,100.0%,$936.14,$899.48,21,$862.82,13
2,2,EQNR,ACTIVE_LEADER,54.6,54.6,0.0,82,Energy,Oil & Gas Integrated,85,13,1,FUND_WEAK,99,+17.2,98%,1.93,100.0%,$33.59,$32.55,595,$31.51,480
5,3,PBR,ACTIVE_LEADER,53.0,54.5,1.5,82,Energy,Oil & Gas Integrated,85,12,0,FUND_MIXED,99,+13.0,83%,1.85,100.0%,$17.60,$17.09,1136,$16.59,987
0,4,CNQ,ACTIVE_LEADER,51.4,54.4,3.0,81,Energy,Oil & Gas E&P,88,12,2,FUND_OK,98,+12.2,85%,1.72,100.0%,$46.31,$45.04,431,$43.77,394
9,5,WDS,ACTIVE_LEADER,48.9,53.9,5.0,81,Energy,Oil & Gas E&P,88,13,2,FUND_WEAK,99,+18.4,95%,1.81,100.0%,$22.34,$21.71,895,$21.09,797
4,6,MPC,ACTIVE_LEADER,48.7,53.7,5.0,86,Energy,Oil & Gas Refining & Marketing,89,12,0,FUND_MIXED,97,+5.6,72%,1.58,100.0%,$221.28,$213.02,90,$204.77,60



  NEW SETUPS — WAIT FOR TRIGGER (2):


,priority_rank,ticker,_status,priority_score,raw_priority_score,concentration_penalty,group_health_score,sector,industry,group_thrust,breakout_quality,failure_risk,fund_flag,rs_percentile,sector_adj_rs,weekly_close_range,ud_volume_ratio,pct_of_52w_high,close,stop_1atr,shares_1atr,stop_2atr,shares_2atr
12,7,CVX,NEW_SETUP,46.9,51.9,5.0,82,Energy,Oil & Gas Integrated,85,12,0,nan,93,-0.2,70%,2.27,100.0%,$189.94,$186.14,105,$182.33,105
20,8,VZ,NEW_SETUP,46.2,46.2,0.0,70,Communication Services,Telecom Services,53,9,1,nan,96,+18.1,86%,2.01,99.8%,$51.12,$50.14,391,$49.17,391



  Full priority pool: 16 candidates scored


In [15]:
if MODE == "weekly":
    # ===========================================================================
    # EXPORT
    # ===========================================================================
    from datetime import date

    tag = date.today().strftime("%Y%m%d")

    # Recompute position sizing to guarantee consistency between CSV and report
    print("Computing final position sizing...")
    for idx, row in winner_features.iterrows():
        if row.get("tier") in ["A", "B+"]:
            s1 = position_size(row["close"], row["stop_1atr"])
            s2 = position_size(row["close"], row["stop_2atr"])
            winner_features.at[idx, "shares_1atr"] = s1["shares"]
            winner_features.at[idx, "position_value_1atr"] = s1["position_value"]
            winner_features.at[idx, "pct_of_account_1atr"] = s1["pct_of_account"]
            winner_features.at[idx, "risk_pct_1atr"] = s1["actual_risk_pct"]
            winner_features.at[idx, "capped_1atr"] = s1["capped"]
            winner_features.at[idx, "shares_2atr"] = s2["shares"]
            winner_features.at[idx, "position_value_2atr"] = s2["position_value"]
            winner_features.at[idx, "pct_of_account_2atr"] = s2["pct_of_account"]
            winner_features.at[idx, "risk_pct_2atr"] = s2["actual_risk_pct"]
            winner_features.at[idx, "capped_2atr"] = s2["capped"]

    # Winner watchlist with full profiles
    winner_export = winners.merge(winner_features, on="ticker", how="left")
    if "master_rank" in winner_export.columns:
        winner_export = winner_export.sort_values("master_rank", ascending=True)
    winner_export.to_csv(f"winners_{tag}.csv", index=False)

    # Priority list
    if "priority_pool" in dir() and len(priority_pool) > 0:
        priority_pool.to_csv(f"priority_{tag}.csv", index=False)

    # Approaching breakout setups
    setup_candidates.to_csv(f"setups_{tag}.csv", index=False)

    # Industry group thrust
    industry_thrust_df.to_csv(f"industry_thrust_{tag}.csv", index=False)

    # Group health
    if "group_health_df" in dir() and len(group_health_df) > 0:
        group_health_df.to_csv(f"group_health_{tag}.csv", index=False)

    # Full universe technical features
    features_df.to_csv(f"universe_features_{tag}.csv", index=False)

    # Commonalities
    commonalities.to_csv(f"commonalities_{tag}.csv", index=False)

    # Market regime
    regime_df.to_csv(f"regime_{tag}.csv", index=False)

    # Text report
    with open(f"report_{tag}.txt", "w") as f:
        f.write(f"Weekly 5%+ Movers Report - {tag}\n")
        f.write("=" * 64 + "\n\n")

        f.write("MARKET REGIME\n")
        f.write("-" * 40 + "\n")
        for k, v in regime.items():
            f.write(f"  {k:>42s} : {v}\n")

        f.write(f"\n\nUNIVERSE: {len(valid)} names\n")
        f.write(f"WINNERS: {len(winner_tickers)} stocks >= {MIN_WEEKLY_RETURN:.0%}\n")
        f.write(f"APPROACHING SETUPS: {len(setup_candidates)} names near breakout\n\n")

        f.write("TECHNICAL COMMONALITIES\n")
        f.write("-" * 40 + "\n")
        f.write(commonalities.to_string(index=False))

        f.write("\n\nSTAGE DISTRIBUTION\n")
        f.write("-" * 40 + "\n")
        for label in ["Winners", "Universe"]:
            df_s = winner_features if label == "Winners" else features_df
            dist = df_s["stage_num"].value_counts().sort_index()
            total = len(df_s)
            parts = []
            for s in [1, 2, 3, 4]:
                n = dist.get(s, 0)
                parts.append(f"S{s}: {100*n/total:.0f}%")
            f.write(f"  {label:>10s}: {' | '.join(parts)}\n")

        f.write("\n\nRS TREND DISTRIBUTION\n")
        f.write("-" * 40 + "\n")
        for label in ["Winners", "Universe"]:
            df_s = winner_features if label == "Winners" else features_df
            dist = df_s["rs_trend"].value_counts()
            total = df_s["rs_trend"].notna().sum()
            parts = []
            for trend in ["Rising", "Flat", "Declining"]:
                n = dist.get(trend, 0)
                pct = 100*n/total if total > 0 else 0
                parts.append(f"{trend}: {n} ({pct:.0f}%)")
            f.write(f"  {label:>10s}: {' | '.join(parts)}\n")

        f.write("\n\nINDUSTRY GROUP THRUST — TOP 15\n")
        f.write("-" * 40 + "\n")
        for _, r in industry_thrust_df.head(15).iterrows():
            f.write(f"  {r['industry']:>40s}  Thrust:{r['thrust_score']:3.0f}  "
                    f"Names:{r['n_names']:2.0f}  "
                    f"Stage2:{r['pct_stage2']:.0f}%  "
                    f"RS:{r['median_rs_pct']:.0f}  "
                    f"Winners:{r['n_winners']:.0f}\n")

        if "group_health_df" in dir() and len(group_health_df) > 0:
            scored_gh = group_health_df[group_health_df["group_health_score"].notna()]
            f.write("\n\nGROUP HEALTH — TOP 15 INDUSTRIES\n")
            f.write("-" * 40 + "\n")
            for _, r in scored_gh.head(15).iterrows():
                f.write(f"  {r['industry']:>40s}  Health:{r['group_health_score']:5.0f}  "
                        f"[{r['group_health_label']:>9s}]  "
                        f"Names:{r['n_names']:2.0f}  "
                        f"S2:{r.get('pct_stage2',0):.0f}%  "
                        f"Leaders:{r.get('leader_density',0):.0%}\n")

        f.write("\n\nSECTOR ETF RS (vs SPY)\n")
        f.write("-" * 40 + "\n")
        for etf in sorted(sector_etf_rs_now.keys(), key=lambda x: sector_etf_rs_now[x], reverse=True):
            f.write(f"  {etf:>6s}: {sector_etf_rs_now[etf]:+.1f}\n")

        f.write(f"\n\nPOSITION SIZING: ${ACCOUNT_SIZE:,.0f} account | "
                f"{RISK_PER_TRADE:.0%} risk/trade | {MAX_POSITION_PCT:.0%} max position\n")
        f.write(f"Dollar risk per trade: ${ACCOUNT_SIZE * RISK_PER_TRADE:,.0f}\n")

            # Priority Review List
        if "priority_pool" in dir() and len(priority_pool) > 0:
            top_p = priority_pool.head(PRIORITY_TOP_N)
            f.write("\n\nPRIORITY REVIEW — TOP {0} NAMES\n".format(PRIORITY_TOP_N))
            f.write("-" * 40 + "\n")
            f.write(f"Regime: {regime.get('REGIME_VERDICT', '?')}\n\n")
            for _, r in top_p.iterrows():
                status = r.get("_status", "")
                ps = r.get("priority_score", 0)
                bq = r.get("breakout_quality", 0)
                fr = r.get("failure_risk", 0)
                ff = r.get("fund_flag", "")
                gt = r.get("group_thrust", 0)
                rs_p = r.get("rs_percentile", 0)
                sa = r.get("sector_adj_rs", 0)
                s1 = r.get("stop_1atr", 0)
                sh1 = r.get("shares_1atr", 0)
                sect = r.get("sector", "")
                pr = r.get("priority_rank", 0)
                f.write(f"  {pr:>2.0f}. {r['ticker']:>6s}  [{status:>14s}]  "
                        f"Score:{ps:5.1f}  BQ:{bq:.0f}  FR:{fr:.0f}  {str(ff):>12s}  "
                        f"RS:{rs_p:.0f}  SAdj:{sa:+.1f}  Thrust:{gt:.0f}  "
                        f"GH:{r.get('group_health_score',0):.0f}[{str(r.get('group_health_label','')):>9s}]  "
                        f"Stop:${s1:.2f}  Shares:{sh1:.0f}  [{sect}]\n")

        f.write("\n\nTIERED WATCHLIST\n")
        f.write("-" * 40 + "\n")
        for tier_name, tier_desc in [("A", "ACTIONABLE"), ("B+", "STAGE 3 WATCH"),
                                      ("B", "WATCHLIST"), ("C", "AVOID")]:
            subset = winner_export[winner_export["tier"] == tier_name].sort_values(
                "master_rank", ascending=True)
            f.write(f"\n  TIER {tier_name} - {tier_desc} ({len(subset)} names):\n")
            if len(subset) > 0:
                for _, r in subset.iterrows():
                    rs_p = r.get('rs_percentile', 0)
                    bq   = r.get('breakout_quality', 0)
                    stg  = r.get('stage_num', '?')
                    pct  = r.get('pct_of_52w_high', 0)
                    bdv  = r.get('breakout_day_vol', 0)
                    ud   = r.get('ud_volume_ratio', 0)
                    wcr  = r.get('weekly_close_range', 0)
                    s1   = r.get('stop_1atr', 0)
                    rs_t = r.get('rs_trend', '')
                    sa_rs = r.get('sector_adj_rs', 0)
                    sect = r.get('sector', '')
                    sh1  = r.get('shares_1atr', 0)
                    pv1  = r.get('position_value_1atr', 0)

                    ff = r.get('fund_flag', '')
                    fr = r.get('failure_risk', 0)
                    pl = r.get('persistence_label', '')
                    # Retrieve ghs and ghl from the current row object 'r'
                    ghs_val = r.get('group_health_score', 0)
                    ghl_val = r.get('group_health_label', '')
                    line = (
                        f"    {r['ticker']:>6s}  [{str(pl):>15s}]  "
                        f"{r['weekly_return']:+.1%}  "
                        f"RS:{rs_p:5.0f} {rs_t:>9s}  "
                        f"SectAdj:{sa_rs:+5.1f}  "
                        f"BQ:{bq:.0f}  FR:{fr:.0f}  {str(ff):>12s}  "
                        f"GH:{ghs_val:.0f}[{str(ghl_val):>9s}]  "
                        f"Stage:{stg}  "
                        f"52wH:{pct:.0%}  "
                        f"BkDayVol:{bdv:.1f}x  "
                        f"U/D:{ud:.1f}  "
                        f"WkCl:{wcr:.0%}  "
                        f"Stop:${s1:.2f}"
                    )

                    # Add position sizing for A and B+ only
                    rp1 = r.get('risk_pct_1atr', 0)
                    capped = r.get('capped_1atr', False)
                    if tier_name in ["A", "B+"] and pd.notna(sh1) and sh1 > 0:
                        cap_flag = " [MAX CAP]" if capped else ""
                        line += f"  -> {sh1:.0f} shares (${pv1:,.0f}, risk {rp1:.2%}){cap_flag}"

                    line += f"  [{sect}]\n"
                    f.write(line)
            else:
                f.write("    (none)\n")

        # Approaching breakout setups
        f.write(f"\n\nAPPROACHING BREAKOUTS - MONDAY WATCHLIST ({len(setup_candidates)} setups)\n")
        f.write("-" * 40 + "\n")
        f.write("Stage 2 + RS > 60 rising + within 3% of 20d pivot + not already broken out + demand\n\n")
        if len(setup_candidates) > 0:
            for _, r in setup_candidates.iterrows():
                rs_p = r.get('rs_percentile', 0)
                bq   = r.get('breakout_quality', 0)
                pct  = r.get('pct_of_52w_high', 0)
                wcr  = r.get('weekly_close_range', 0)
                rs_t = r.get('rs_trend', '')
                sa_rs = r.get('sector_adj_rs', 0)
                sect = r.get('sector', '')
                s1   = r.get('stop_1atr', 0)
                sh1  = r.get('shares_1atr', 0)
                pv1  = r.get('position_value_1atr', 0)
                ud   = r.get('ud_volume_ratio', 0)

                ff = r.get('fund_flag', '')
                fr = r.get('failure_risk', 0)
                pl = r.get('persistence_label', '')
                # Retrieve ghs and ghl from the current row object 'r'
                ghs_val = r.get('group_health_score', 0)
                ghl_val = r.get('group_health_label', '')
                line = (
                    f"    {r['ticker']:>6s}  [{str(pl):>15s}]  "
                    f"${r['close']:.2f}  "
                    f"RS:{rs_p:5.0f} {rs_t:>9s}  "
                    f"SectAdj:{sa_rs:+5.1f}  "
                    f"BQ:{bq:.0f}  FR:{fr:.0f}  {str(ff):>12s}  "
                    f"GH:{ghs_val:.0f}[{str(ghl_val):>9s}]  "
                    f"52wH:{pct:.0%}  "
                    f"U/D:{ud:.1f}  "
                    f"WkCl:{wcr:.0%}  "
                    f"Stop:${s1:.2f}"
                )
                if pd.notna(sh1) and sh1 > 0:
                    line += f"  -> {sh1:.0f} shares (${pv1:,.0f})"
                line += f"  [{sect}]\n"
                f.write(line)
        else:
            f.write("  (no setups found)\n")

    # ===========================================================================
    # SAVE WEEKLY SNAPSHOTS TO GOOGLE DRIVE
    # ===========================================================================
    snap_date = RUN_DATE.strftime("%Y%m%d")
    iso_year, iso_week, _ = RUN_DATE.isocalendar()
    snap_week_tag = f"{iso_year}W{iso_week:02d}"

    # Ticker snapshot: all winners + setups with key columns
    ticker_snap_cols = ["ticker", "tier", "breakout_quality", "failure_risk", "stage_num",
                        "rs_percentile", "mansfield_rs", "sector", "industry",
                        "group_thrust", "fund_flag", "pct_of_52w_high", "sector_adj_rs",
                        "weekly_return", "close"]
    ticker_snap_cols = [c for c in ticker_snap_cols if c in winner_features.columns]

    snap_winners = winner_features[ticker_snap_cols].copy()
    snap_winners["run_date"] = RUN_DATE
    snap_winners["model_version"] = MODEL_VERSION
    snap_winners["iso_week"] = snap_week_tag
    snap_winners["status"] = "winner"

    # Add setups
    setup_snap_cols = [c for c in ticker_snap_cols if c in setup_candidates.columns]
    snap_setups = setup_candidates[setup_snap_cols].copy()
    snap_setups["run_date"] = RUN_DATE
    snap_setups["model_version"] = MODEL_VERSION
    snap_setups["iso_week"] = snap_week_tag
    snap_setups["status"] = "setup"

    snap_all = pd.concat([snap_winners, snap_setups], ignore_index=True)
    snap_path = os.path.join(SNAP_DIR, f"snap_{snap_week_tag}_{snap_date}.csv")
    snap_all.to_csv(snap_path, index=False)
    print(f"Saved ticker snapshot: {snap_path} ({len(snap_all)} rows)")

    # Group snapshot
    if len(industry_thrust_df) > 0:
        group_snap = industry_thrust_df.copy()
        group_snap["run_date"] = RUN_DATE
        group_snap["model_version"] = MODEL_VERSION
        group_snap["iso_week"] = snap_week_tag
        group_path = os.path.join(GROUP_DIR, f"group_{snap_week_tag}_{snap_date}.csv")
        group_snap.to_csv(group_path, index=False)
        print(f"Saved group snapshot: {group_path} ({len(group_snap)} rows)")

    print(f"\nExported:")
    print(f"  winners_{tag}.csv           - winner profiles + tiers + sizing")
    print(f"  setups_{tag}.csv            - approaching breakout candidates")
    print(f"  universe_features_{tag}.csv - full universe technicals")
    print(f"  commonalities_{tag}.csv     - winners vs universe comparison")
    print(f"  regime_{tag}.csv            - market regime snapshot")
    print(f"  report_{tag}.txt            - text summary report")

Computing final position sizing...
Saved ticker snapshot: /content/drive/MyDrive/leadership_monitor/snapshots/snap_2026W10_20260308.csv (82 rows)
Saved group snapshot: /content/drive/MyDrive/leadership_monitor/group_snapshots/group_2026W10_20260308.csv (13 rows)

Exported:
  winners_20260308.csv           - winner profiles + tiers + sizing
  setups_20260308.csv            - approaching breakout candidates
  universe_features_20260308.csv - full universe technicals
  commonalities_20260308.csv     - winners vs universe comparison
  regime_20260308.csv            - market regime snapshot
  report_20260308.txt            - text summary report


In [16]:
# ===========================================================================
# DAILY MONITOR MODE
# ===========================================================================
# Run this cell on Mon-Fri to get a short change-detection report
# comparing today's data against the last weekly baseline.
# This cell is OPTIONAL — skip it on the weekly full run.
#
# To use: run Cells 0-8 (universe + profiling), then skip to this cell.
# It loads the weekly baseline from Drive and produces a delta report.
# ===========================================================================

# This cell runs automatically in daily mode

if MODE == "daily":
    print("=" * 70)
    print("  DAILY MONITOR — CHANGE DETECTION vs WEEKLY BASELINE")
    print("=" * 70)

    # Load most recent weekly snapshot as baseline
    snap_files = sorted([f for f in os.listdir(SNAP_DIR) if f.endswith('.csv')], reverse=True)
    baseline = pd.DataFrame()
    baseline_date = None

    for f in snap_files:
        try:
            df = pd.read_csv(os.path.join(SNAP_DIR, f))
            if "model_version" in df.columns and (df["model_version"] == MODEL_VERSION).any():
                baseline = df[df["model_version"] == MODEL_VERSION]
                baseline_date = f.replace("snap_", "").replace(".csv", "")
                break
        except Exception:
            pass

    if baseline.empty:
        print("\n  No baseline found. Run the full weekly script first.")
    else:
        print(f"\n  Baseline: {baseline_date} ({len(baseline)} names)")
        baseline_winners = baseline[baseline["status"] == "winner"]
        baseline_setups = baseline[baseline["status"] == "setup"]
        baseline_a = set(baseline_winners[baseline_winners["tier"] == "A"]["ticker"].tolist())
        baseline_bplus = set(baseline_winners[baseline_winners["tier"] == "B+"]["ticker"].tolist())
        baseline_b = set(baseline_winners[baseline_winners["tier"] == "B"]["ticker"].tolist())
        baseline_tracked = baseline_a | baseline_bplus | baseline_b
        baseline_setup_tickers = set(baseline_setups["ticker"].tolist())

        # Current state from today's profiling (features_df from Cell 7)
        today_close = close[valid].iloc[-1]
        today_ret_1d = (close[valid].iloc[-1] / close[valid].iloc[-2] - 1) if len(close) >= 2 else pd.Series()

        # ── 1. NEW MOVERS (crossed 5% threshold since baseline) ──
        if len(close) >= 6:
            current_5d_ret = close[valid].iloc[-1] / close[valid].iloc[-6] - 1
            new_movers = current_5d_ret[current_5d_ret >= MIN_WEEKLY_RETURN]
            new_movers = new_movers[~new_movers.index.isin(baseline_winners["ticker"].tolist())]
            if len(new_movers) > 0:
                print(f"\n  NEW 5%+ MOVERS ({len(new_movers)} names):")
                for t in new_movers.sort_values(ascending=False).head(10).index:
                    feat = features_df[features_df["ticker"] == t]
                    if len(feat) > 0:
                        r = feat.iloc[0]
                        print(f"    {t:>6s}  +{new_movers[t]:.1%}  "
                              f"RS:{r.get('rs_percentile',0):5.0f}  "
                              f"Stage:{r.get('stage_num','?')}  "
                              f"52wH:{r.get('pct_of_52w_high',0):.0%}  "
                              f"[{r.get('sector','')}]")
            else:
                print(f"\n  NEW 5%+ MOVERS: None")

        # ── 2. TRACKED NAMES: STATUS CHECK ──
        print(f"\n  TRACKED NAME STATUS (A/B+/B from baseline):")
        for t in sorted(baseline_tracked):
            feat = features_df[features_df["ticker"] == t]
            base_row = baseline_winners[baseline_winners["ticker"] == t]
            if len(feat) == 0 or len(base_row) == 0:
                continue
            r = feat.iloc[0]
            b = base_row.iloc[0]

            old_tier = b.get("tier", "?")
            old_bq = b.get("breakout_quality", 0)
            new_bq = r.get("breakout_quality", 0)
            bq_delta = new_bq - old_bq if pd.notna(new_bq) and pd.notna(old_bq) else 0

            # Volume spike today?
            vol_spike = ""
            if t in volume.columns and len(volume) >= 2:
                today_vol = volume[t].iloc[-1]
                avg_vol = volume[t].iloc[-50:].mean() if len(volume) >= 50 else volume[t].mean()
                if pd.notna(today_vol) and pd.notna(avg_vol) and avg_vol > 0:
                    vol_ratio = today_vol / avg_vol
                    if vol_ratio >= 2.0:
                        vol_spike = f" VOL:{vol_ratio:.1f}x!"

            # 1-day return
            ret_1d = today_ret_1d.get(t, np.nan)
            ret_str = f"{ret_1d:+.1%}" if pd.notna(ret_1d) else "N/A"

            # Status flags
            flags = []
            if bq_delta >= 2:
                flags.append("IMPROVING")
            elif bq_delta <= -2:
                flags.append("DETERIORATING")

            # Near stop?
            stop = r.get("stop_1atr", np.nan)
            cur_price = r.get("close", np.nan)
            if pd.notna(stop) and pd.notna(cur_price) and cur_price > 0:
                stop_dist = (cur_price - stop) / cur_price
                if stop_dist < 0.02:
                    flags.append("NEAR STOP")
                if stop_dist < 0:
                    flags.append("BELOW STOP!")

            flag_str = "  ".join(flags) if flags else ""
            print(f"    {t:>6s}  [{old_tier:>2s}]  Today:{ret_str:>6s}{vol_spike}  "
                  f"BQ:{new_bq:.0f}({bq_delta:+.0f})  "
                  f"RS:{r.get('rs_percentile',0):.0f}  "
                  f"Stage:{r.get('stage_num','?')}  {flag_str}")

        # ── 3. SETUP TRIGGERS ──
        print(f"\n  SETUP TRIGGER CHECK:")
        triggered = 0
        for t in sorted(baseline_setup_tickers):
            feat = features_df[features_df["ticker"] == t]
            if len(feat) == 0:
                continue
            r = feat.iloc[0]
            # Did it cross 5%+ since baseline?
            if len(close) >= 6:
                ret_5d = close[valid].iloc[-1] / close[valid].iloc[-6] - 1
                if t in ret_5d.index and ret_5d[t] >= MIN_WEEKLY_RETURN:
                    triggered += 1
                    vol_ratio_str = ""
                    if t in volume.columns and len(volume) >= 50:
                        today_vol = volume[t].iloc[-1]
                        avg_vol = volume[t].iloc[-50:].mean()
                        if pd.notna(today_vol) and pd.notna(avg_vol) and avg_vol > 0:
                            vol_ratio_str = f" Vol:{today_vol/avg_vol:.1f}x"
                    print(f"    {t:>6s}  TRIGGERED! +{ret_5d[t]:.1%}{vol_ratio_str}  "
                          f"RS:{r.get('rs_percentile',0):.0f}  [{r.get('sector','')}]")
        if triggered == 0:
            print(f"    No setups triggered yet.")

        # ── 4. INDUSTRY GROUP CHANGES ──
        group_files_sorted = sorted([f for f in os.listdir(GROUP_DIR) if f.endswith('.csv')], reverse=True)
        if group_files_sorted:
            try:
                last_group = pd.read_csv(os.path.join(GROUP_DIR, group_files_sorted[0]))
                last_group = last_group[last_group["model_version"] == MODEL_VERSION] if "model_version" in last_group.columns else last_group
                if len(last_group) > 0 and len(industry_thrust_df) > 0:
                    print(f"\n  INDUSTRY GROUP CHANGES vs {group_files_sorted[0]}:")
                    merged_g = industry_thrust_df.merge(
                        last_group[["industry", "thrust_score"]],
                        on="industry", how="outer", suffixes=("_now", "_prev"))
                    merged_g["delta"] = merged_g["thrust_score_now"].fillna(0) - merged_g["thrust_score_prev"].fillna(0)
                    improving = merged_g[merged_g["delta"] > 3].sort_values("delta", ascending=False)
                    weakening = merged_g[merged_g["delta"] < -3].sort_values("delta")

                    if len(improving) > 0:
                        print(f"    Strengthening:")
                        for _, r in improving.head(5).iterrows():
                            print(f"      {r['industry']:>40s}: {r.get('thrust_score_prev',0):.0f} -> {r.get('thrust_score_now',0):.0f} ({r['delta']:+.0f})")
                    if len(weakening) > 0:
                        print(f"    Weakening:")
                        for _, r in weakening.head(5).iterrows():
                            print(f"      {r['industry']:>40s}: {r.get('thrust_score_prev',0):.0f} -> {r.get('thrust_score_now',0):.0f} ({r['delta']:+.0f})")
                    if len(improving) == 0 and len(weakening) == 0:
                        print(f"    No significant group changes.")
            except Exception as e:
                print(f"    Could not compare groups: {e}")

        print(f"\n{'='*70}")
        print(f"  Monitor complete. Full weekly rebuild recommended on weekends.")
        print(f"{'='*70}")

        # Save daily monitor report
        from datetime import date as _date
        monitor_tag = _date.today().strftime("%Y%m%d")
        monitor_path = f"daily_monitor_{monitor_tag}.txt"
        with open(monitor_path, "w") as mf:
            mf.write(f"Daily Monitor Report - {monitor_tag}\n")
            mf.write(f"Baseline: {baseline_date}\n")
            mf.write("=" * 60 + "\n\n")

            # Regime summary
            mf.write(f"REGIME: {regime.get('REGIME_VERDICT', '?')}\n")
            mf.write(f"SPY: {regime.get('SPY_close', '?')} | RSI: {regime.get('SPY_RSI14', '?')}\n")
            mf.write(f"Breadth: {regime.get('pct_above_50sma', '?')} > 50sma | {regime.get('pct_above_200sma', '?')} > 200sma\n\n")

            # New movers
            if len(close) >= 6:
                current_5d_ret_all = close[valid].iloc[-1] / close[valid].iloc[-6] - 1
                new_movers_all = current_5d_ret_all[current_5d_ret_all >= MIN_WEEKLY_RETURN]
                new_movers_all = new_movers_all[~new_movers_all.index.isin(baseline_winners["ticker"].tolist())]
                mf.write(f"NEW 5%+ MOVERS: {len(new_movers_all)}\n")
                for t in new_movers_all.sort_values(ascending=False).head(15).index:
                    feat = features_df[features_df["ticker"] == t]
                    if len(feat) > 0:
                        r = feat.iloc[0]
                        mf.write(f"  {t:>6s}  +{new_movers_all[t]:.1%}  RS:{r.get('rs_percentile',0):5.0f}  Stage:{r.get('stage_num','?')}  [{r.get('sector','')}]\n")

            # Tracked names
            mf.write(f"\nTRACKED NAME STATUS:\n")
            for t in sorted(baseline_tracked):
                feat = features_df[features_df["ticker"] == t]
                base_row = baseline_winners[baseline_winners["ticker"] == t]
                if len(feat) == 0 or len(base_row) == 0:
                    continue
                r = feat.iloc[0]
                b = base_row.iloc[0]
                old_tier = b.get("tier", "?")
                ret_1d = today_ret_1d.get(t, 0) if not today_ret_1d.empty else 0
                new_bq = r.get("breakout_quality", 0)
                old_bq = b.get("breakout_quality", 0)
                bq_d = new_bq - old_bq if pd.notna(new_bq) and pd.notna(old_bq) else 0
                mf.write(f"  {t:>6s}  [{old_tier:>2s}]  {ret_1d:+.1%}  BQ:{new_bq:.0f}({bq_d:+.0f})  RS:{r.get('rs_percentile',0):.0f}\n")

            # Setup triggers
            mf.write(f"\nSETUP TRIGGERS:\n")
            triggered_count = 0
            if len(close) >= 6:
                ret_5d_check = close[valid].iloc[-1] / close[valid].iloc[-6] - 1
                for t in sorted(baseline_setup_tickers):
                    if t in ret_5d_check.index and ret_5d_check[t] >= MIN_WEEKLY_RETURN:
                        triggered_count += 1
                        mf.write(f"  {t:>6s}  TRIGGERED +{ret_5d_check[t]:.1%}\n")
            if triggered_count == 0:
                mf.write(f"  (none)\n")

        print(f"\nSaved: {monitor_path}")

        # Download
        try:
            from google.colab import files as _files
            _files.download(monitor_path)
        except Exception:
            print(f"  (auto-download not available, file saved locally)")
else:
    print("Skipping daily monitor (MODE = weekly).")


Skipping daily monitor (MODE = weekly).


In [17]:
if MODE == "weekly":
    # ===========================================================================
    # DOWNLOAD ALL FILES
    # ===========================================================================
    from google.colab import files

    for f in [
        f"winners_{tag}.csv",
        f"setups_{tag}.csv",
        f"universe_features_{tag}.csv",
        f"commonalities_{tag}.csv",
        f"regime_{tag}.csv",
        f"report_{tag}.txt",
    f"priority_{tag}.csv",
        f"industry_thrust_{tag}.csv",
    f"group_health_{tag}.csv",
    ]:
        try:
            files.download(f)
        except Exception as e:
            print(f"Could not download {f}: {e}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>